<a href="https://colab.research.google.com/github/hj245668/ds6_warpUp/blob/main/1117_wrapup_3t.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

warpUp : 워렌버핏따라잡기
-



20251117 target : Margin of Safty



In [ ]:
"""
    "NCSoft": "036570.KS",   # 엔씨소프트
    "Krafton": "259960.KS",  # 크래프톤
    "Mgame": "058630.KQ",    # 엠게임 (KOSDAQ)
"""

In [ ]:
"""
import this             # let's do more of those!
"""

In [ ]:
!pip install finance-datareader plotly --quiet

import FinanceDataReader as fdr
import pandas as pd
import plotly.express as px

# 1) 코드 매핑
codes = {
    "NCSoft": "036570",   # 엔씨소프트 (코스피)
    "Krafton": "259960",  # 크래프톤 (코스피)
    "Mgame": "058630",    # 엠게임 (코스닥)
}

start = "2020-01-01"  # 시작 날짜

price_dict = {}
for name, code in codes.items():
    print(f"{name} ({code}) 다운로드 중...")
    df = fdr.DataReader(code, start)
    price_dict[name] = df

# 2) 종가만 모아서 하나의 DataFrame으로
close_df = pd.DataFrame({
    name: df["Close"] for name, df in price_dict.items()
})

close_df.tail()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.3 MB/s eta 0:00:00
NCSoft (036570) 다운로드 중...
Krafton (259960) 다운로드 중...
Mgame (058630) 다운로드 중...


,NCSoft,Krafton,Mgame
Date,,,
2025-11-11,229500,264000.0,6230
2025-11-12,242000,272000.0,6350
2025-11-13,233000,275000.0,6540
2025-11-14,231500,270000.0,6460
2025-11-17,227000,267000.0,6280


In [ ]:
# index를 칼럼으로 빼서 Plotly용 long-form으로 변환
df_plot = close_df.reset_index().rename(columns={"index": "Date"})

fig = px.line(
    df_plot,
    x="Date",
    y=["NCSoft", "Krafton", "Mgame"],
    title="엔씨소프트 / 크래프톤 / 엠게임 일별 종가",
    labels={"value": "종가 (KRW)", "variable": "종목", "Date": "날짜"},
)
fig.update_layout(legend_title_text="종목")
fig.show()


In [ ]:
# 첫 날 가격을 100으로 맞춰서 상대 수익률 비교
# 2020년 1월에 100만원을 샀다면...지금 얼마?

norm_df = close_df / close_df.iloc[0] * 100
norm_plot = norm_df.reset_index().rename(columns={"index": "Date"})

fig_norm = px.line(
    norm_plot,
    x="Date",
    y=["NCSoft", "Krafton", "Mgame"],
    title="엔씨소프트 / 크래프톤 / 엠게임 수익률 흐름 (첫 날 = 100)",
    labels={"value": "지수화된 가격", "variable": "종목", "Date": "날짜"},
)
fig_norm.update_layout(legend_title_text="종목")
fig_norm.show()


In [ ]:
nc = price_dict["NCSoft"].copy()
nc_ma = nc[["Close"]].copy()
nc_ma["MA20"] = nc_ma["Close"].rolling(window=20).mean()
nc_ma["MA60"] = nc_ma["Close"].rolling(window=60).mean()
nc_ma = nc_ma.reset_index().rename(columns={"index": "Date"})

fig_nc = px.line(
    nc_ma,
    x="Date",
    y=["Close", "MA20", "MA60"],
    title="엔씨소프트 종가 및 20/60일 이동평균",
    labels={"value": "가격 (KRW)", "variable": "구분", "Date": "날짜"},
)
fig_nc.update_layout(legend_title_text="구분")
fig_nc.show()


한국 게임주 3종 (엔씨소프트, 크래프톤, 엠게임)에 대해
NLP + 가격 데이터로 “N일 후 주가 방향”을 예측할 수 있는 구조 만들기

NCSoft (036570)
Krafton (259960)
Mgame (058630)


Feature를 정해보자

▼ ▼▼▼▼▼▼▼▼▼▼  최강 크롤링 만들기 - 병렬검색, 형식 구애 최소화,

In [ ]:
# ========================================
# 1단계: 필요한 라이브러리 설치 및 임포트
# ========================================
!pip install requests beautifulsoup4 pandas tqdm -q

import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import time
import re
from urllib.parse import quote

# ========================================
# 2단계: 키워드 정의
# ========================================
sector_trend_keywords = {
    "호재": [
        "신작 출시", "출시 임박", "흥행", "매출 1위", "사전예약",
        "중국 판호", "판호 발급", "글로벌 론칭", "다운로드 1위", "유저수 증가",
        "IP 확장", "콘솔 진출", "멀티 플랫폼", "라이브 서비스",
        "인공지능 NPC", "AI 기술 적용", "그래픽 업그레이드"
    ],
    "악재": [
        "흥행 부진", "매출 감소", "유저 이탈", "과금 논란", "확률형 아이템 논란",
        "규제 강화", "판호 불허", "심의 보류",
        "점검 논란", "서버 장애", "오류 발생",
        "리뷰 폭격", "평점 하락", "환불 사태"
    ]
}

company_keywords = {
    "NCSoft": {
        "company_terms": ["엔씨소프트", "NC소프트", "NCSoft"],
        "game_terms": [
            "리니지", "리니지M", "리니지W", "리니지2M",
            "아이온", "아이온2", "블레이드앤소울", "블레이드 & 소울",
            "프로젝트 LLL", "LLL"
        ],
    },
    "Krafton": {
        "company_terms": ["크래프톤", "KRAFTON"],
        "game_terms": [
            "배틀그라운드", "배그", "PUBG", "PUBG MOBILE",
            "다크앤다커", "Dark and Darker", "Dark and Darker Mobile",
            "inZOI", "인조이"
        ],
    },
    "Mgame": {
        "company_terms": ["엠게임", "Mgame"],
        "game_terms": [
            "열혈강호", "열혈강호 온라인", "열혈강호M",
            "나이트 온라인", "Knight Online"
        ],
    },
}

# ========================================
# 3단계: 네이버 뉴스 크롤링 함수
# ========================================
def crawl_naver_news(keyword, max_pages=3):
    """네이버 증권 뉴스 크롤링"""
    articles = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    for page in range(1, max_pages + 1):
        try:
            url = f"https://search.naver.com/search.naver?where=news&query={quote(keyword)}&sm=tab_opt&sort=1&photo=0&field=0&pd=0&ds=&de=&docid=&related=0&mynews=0&office_type=0&office_section_code=0&news_office_checked=&nso=so:dd,p:all,a:all&start={1 + (page-1)*10}"

            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')

            news_items = soup.select('div.news_area')

            for item in news_items:
                try:
                    title_elem = item.select_one('a.news_tit')
                    title = title_elem.text.strip() if title_elem else ""
                    link = title_elem['href'] if title_elem else ""

                    desc_elem = item.select_one('div.news_dsc')
                    description = desc_elem.text.strip() if desc_elem else ""

                    info_elem = item.select_one('div.info_group')
                    press = info_elem.select_one('a.info.press').text.strip() if info_elem and info_elem.select_one('a.info.press') else ""

                    date_elem = item.select_one('span.info')
                    date = date_elem.text.strip() if date_elem else ""

                    articles.append({
                        'source': '네이버',
                        'keyword': keyword,
                        'title': title,
                        'description': description,
                        'press': press,
                        'date': date,
                        'link': link,
                        'crawled_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    })
                except Exception as e:
                    continue

            time.sleep(0.3)  # 요청 간격

        except Exception as e:
            print(f"네이버 크롤링 오류 ({keyword}, page {page}): {e}")
            continue

    return articles

# ========================================
# 4단계: 다음 뉴스 크롤링 함수
# ========================================
def crawl_daum_news(keyword, max_pages=3):
    """다음 증권 뉴스 크롤링"""
    articles = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    for page in range(1, max_pages + 1):
        try:
            url = f"https://search.daum.net/search?w=news&q={quote(keyword)}&sort=recency&p={page}"

            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')

            news_items = soup.select('div.news_wrap')

            for item in news_items:
                try:
                    title_elem = item.select_one('a.tit_main')
                    title = title_elem.text.strip() if title_elem else ""
                    link = title_elem['href'] if title_elem else ""

                    desc_elem = item.select_one('p.desc')
                    description = desc_elem.text.strip() if desc_elem else ""

                    press_elem = item.select_one('span.txt_info')
                    press = press_elem.text.strip() if press_elem else ""

                    date_elem = item.select_one('span.gem_txt')
                    date = date_elem.text.strip() if date_elem else ""

                    articles.append({
                        'source': '다음',
                        'keyword': keyword,
                        'title': title,
                        'description': description,
                        'press': press,
                        'date': date,
                        'link': link,
                        'crawled_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    })
                except Exception as e:
                    continue

            time.sleep(0.3)

        except Exception as e:
            print(f"다음 크롤링 오류 ({keyword}, page {page}): {e}")
            continue

    return articles

# ========================================
# 5단계: 병렬 처리로 빠른 크롤링
# ========================================
def crawl_all_keywords(keywords_list, max_pages=3, max_workers=10):
    """모든 키워드를 병렬로 크롤링"""
    all_articles = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []

        # 네이버와 다음 모두 크롤링
        for keyword in keywords_list:
            futures.append(executor.submit(crawl_naver_news, keyword, max_pages))
            futures.append(executor.submit(crawl_daum_news, keyword, max_pages))

        # 진행상황 표시
        for future in tqdm(as_completed(futures), total=len(futures), desc="크롤링 진행"):
            try:
                result = future.result()
                all_articles.extend(result)
            except Exception as e:
                print(f"오류 발생: {e}")
                continue

    return all_articles

# ========================================
# 6단계: 메인 실행 함수
# ========================================
def main():
    print("="*60)
    print("🚀 증권 뉴스 크롤링 시작")
    print("="*60)

    # 모든 키워드 수집
    all_keywords = []

    # 섹터 트렌드 키워드
    for category, keywords in sector_trend_keywords.items():
        all_keywords.extend(keywords)

    # 기업 및 게임 키워드
    for company, terms in company_keywords.items():
        all_keywords.extend(terms['company_terms'])
        all_keywords.extend(terms['game_terms'])

    print(f"📊 총 {len(all_keywords)}개 키워드 크롤링 예정")
    print(f"🔍 키워드 예시: {all_keywords[:5]}")
    print()

    # 크롤링 실행 (페이지 수와 동시 작업 수 조정 가능)
    start_time = time.time()
    articles = crawl_all_keywords(all_keywords, max_pages=2, max_workers=15)
    end_time = time.time()

    print()
    print("="*60)
    print(f"✅ 크롤링 완료!")
    print(f"⏱️  소요 시간: {end_time - start_time:.2f}초")
    print(f"📰 수집된 기사 수: {len(articles)}개")
    print("="*60)

    # DataFrame 생성
    if articles:
        df = pd.DataFrame(articles)

        # 중복 제거 (링크 기준)
        df = df.drop_duplicates(subset=['link'], keep='first')
        print(f"🔄 중복 제거 후: {len(df)}개 기사")

        # CSV 저장
        filename = f"news_crawling_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"💾 저장 완료: {filename}")

        # 미리보기
        print("\n📋 데이터 미리보기:")
        print(df.head())

        return df
    else:
        print("⚠️  수집된 기사가 없습니다.")
        return None

# ========================================
# 7단계: 실행
# ========================================
if __name__ == "__main__":
    df_result = main()

🚀 증권 뉴스 크롤링 시작
📊 총 62개 키워드 크롤링 예정
🔍 키워드 예시: ['신작 출시', '출시 임박', '흥행', '매출 1위', '사전예약']



크롤링 진행:   0%|          | 0/124 [00:00<?, ?it/s]


✅ 크롤링 완료!
⏱️  소요 시간: 23.47초
📰 수집된 기사 수: 0개
⚠️  수집된 기사가 없습니다.


In [ ]:
# ========================================
# 개선된 네이버+다음 뉴스 크롤링 (2025년 버전)
# ========================================

# 1단계: 라이브러리 설치
!pip install requests beautifulsoup4 pandas tqdm lxml -q

import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import time
from urllib.parse import quote
import warnings
warnings.filterwarnings('ignore')

# ========================================
# 2단계: 키워드 정의
# ========================================
sector_trend_keywords = {
    "호재": [
        "신작 출시", "출시 임박", "흥행", "매출 1위", "사전예약",
        "중국 판호", "판호 발급", "글로벌 론칭", "다운로드 1위", "유저수 증가",
        "IP 확장", "콘솔 진출", "멀티 플랫폼", "라이브 서비스",
        "인공지능 NPC", "AI 기술 적용", "그래픽 업그레이드"
    ],
    "악재": [
        "흥행 부진", "매출 감소", "유저 이탈", "과금 논란", "확률형 아이템 논란",
        "규제 강화", "판호 불허", "심의 보류",
        "점검 논란", "서버 장애", "오류 발생",
        "리뷰 폭격", "평점 하락", "환불 사태"
    ]
}

company_keywords = {
    "NCSoft": {
        "company_terms": ["엔씨소프트", "NC소프트", "NCSoft"],
        "game_terms": [
            "리니지", "리니지M", "리니지W", "리니지2M",
            "아이온", "아이온2", "블레이드앤소울", "블레이드 & 소울",
            "프로젝트 LLL", "LLL"
        ],
    },
    "Krafton": {
        "company_terms": ["크래프톤", "KRAFTON"],
        "game_terms": [
            "배틀그라운드", "배그", "PUBG", "PUBG MOBILE",
            "다크앤다커", "Dark and Darker", "Dark and Darker Mobile",
            "inZOI", "인조이"
        ],
    },
    "Mgame": {
        "company_terms": ["엠게임", "Mgame"],
        "game_terms": [
            "열혈강호", "열혈강호 온라인", "열혈강호M",
            "나이트 온라인", "Knight Online"
        ],
    },
}

# ========================================
# 3단계: 테스트용 HTML 구조 확인 함수
# ========================================
def test_naver_structure(keyword):
    """네이버 뉴스 HTML 구조 확인 (디버깅용)"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    url = f"https://search.naver.com/search.naver?where=news&query={quote(keyword)}&sort=1"

    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')

        # 여러 가능한 선택자 테스트
        selectors = [
            'div.news_wrap',
            'div.news_area',
            'div.news_wrap.api_ani_send',
            'li.bx',
            'div.group_news'
        ]

        print(f"\n키워드: {keyword}")
        print("="*60)
        for sel in selectors:
            items = soup.select(sel)
            print(f"{sel}: {len(items)}개 발견")

            if items and len(items) > 0:
                print(f"✅ 사용 가능한 선택자: {sel}")
                # 첫 번째 항목의 하위 구조 확인
                first_item = items[0]
                print("\n하위 구조:")
                title = first_item.select_one('a.news_tit, a.tit')
                desc = first_item.select_one('div.dsc_wrap, div.news_dsc')
                print(f"  - 제목: {title.text[:30] if title else 'None'}...")
                print(f"  - 설명: {'있음' if desc else 'None'}")
                return sel

        print("\n⚠️ 작동하는 선택자를 찾지 못했습니다.")
        # HTML 일부 출력
        print("\nHTML 일부:")
        print(soup.prettify()[:1000])

    except Exception as e:
        print(f"오류: {e}")

    return None

# ========================================
# 4단계: 개선된 네이버 크롤링 함수
# ========================================
def crawl_naver_news_v2(keyword, max_pages=3):
    """개선된 네이버 뉴스 크롤링 (여러 선택자 시도)"""
    articles = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': 'https://www.naver.com/'
    }

    for page in range(1, max_pages + 1):
        try:
            start = 1 + (page - 1) * 10
            url = f"https://search.naver.com/search.naver?where=news&query={quote(keyword)}&sort=1&start={start}"

            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code != 200:
                print(f"⚠️ 네이버 응답 오류 ({keyword}, page {page}): {response.status_code}")
                continue

            soup = BeautifulSoup(response.text, 'lxml')

            # 여러 선택자 시도 (2024-2025년 네이버 구조 대응)
            news_items = None
            possible_selectors = [
                'div.news_wrap.api_ani_send',
                'div.news_wrap',
                'div.news_area',
                'li.bx',
                'div.group_news > ul > li'
            ]

            for selector in possible_selectors:
                items = soup.select(selector)
                if items and len(items) > 0:
                    news_items = items
                    break

            if not news_items:
                print(f"⚠️ 뉴스 항목을 찾을 수 없음 ({keyword}, page {page})")
                continue

            for item in news_items:
                try:
                    # 제목과 링크 (여러 가능성 시도)
                    title_elem = item.select_one('a.news_tit') or item.select_one('a.tit')
                    if not title_elem:
                        continue

                    title = title_elem.get_text(strip=True)
                    link = title_elem.get('href', '')

                    # 설명
                    desc_elem = (item.select_one('div.news_dsc') or
                               item.select_one('div.dsc_wrap') or
                               item.select_one('a.api_txt_lines'))
                    description = desc_elem.get_text(strip=True) if desc_elem else ""

                    # 언론사
                    press_elem = (item.select_one('a.info.press') or
                                item.select_one('span.info.press') or
                                item.select_one('a.press'))
                    press = press_elem.get_text(strip=True) if press_elem else ""

                    # 날짜
                    date_elem = item.select_one('span.info')
                    date = date_elem.get_text(strip=True) if date_elem else ""

                    articles.append({
                        'source': '네이버',
                        'keyword': keyword,
                        'title': title,
                        'description': description,
                        'press': press,
                        'date': date,
                        'link': link,
                        'crawled_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    })

                except Exception as e:
                    continue

            time.sleep(0.5)  # 안전한 크롤링을 위한 대기

        except Exception as e:
            print(f"네이버 크롤링 오류 ({keyword}, page {page}): {e}")
            continue

    return articles

# ========================================
# 5단계: 개선된 다음 크롤링 함수
# ========================================
def crawl_daum_news_v2(keyword, max_pages=3):
    """개선된 다음 뉴스 크롤링"""
    articles = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': 'https://www.daum.net/'
    }

    for page in range(1, max_pages + 1):
        try:
            url = f"https://search.daum.net/search?w=news&q={quote(keyword)}&sort=recency&p={page}"

            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code != 200:
                print(f"⚠️ 다음 응답 오류 ({keyword}, page {page}): {response.status_code}")
                continue

            soup = BeautifulSoup(response.text, 'lxml')

            # 다음 뉴스 구조
            news_items = soup.select('div.news_wrap') or soup.select('c-doc')

            if not news_items:
                print(f"⚠️ 뉴스 항목을 찾을 수 없음 - 다음 ({keyword}, page {page})")
                continue

            for item in news_items:
                try:
                    title_elem = item.select_one('a.tit_main') or item.select_one('a.link_txt')
                    if not title_elem:
                        continue

                    title = title_elem.get_text(strip=True)
                    link = title_elem.get('href', '')

                    desc_elem = item.select_one('p.desc') or item.select_one('p.f_eb')
                    description = desc_elem.get_text(strip=True) if desc_elem else ""

                    press_elem = item.select_one('span.txt_info') or item.select_one('span.f_nb')
                    press = press_elem.get_text(strip=True) if press_elem else ""

                    date_elem = item.select_one('span.gem_txt') or item.select_one('span.date')
                    date = date_elem.get_text(strip=True) if date_elem else ""

                    articles.append({
                        'source': '다음',
                        'keyword': keyword,
                        'title': title,
                        'description': description,
                        'press': press,
                        'date': date,
                        'link': link,
                        'crawled_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    })

                except Exception as e:
                    continue

            time.sleep(0.5)

        except Exception as e:
            print(f"다음 크롤링 오류 ({keyword}, page {page}): {e}")
            continue

    return articles

# ========================================
# 6단계: 병렬 크롤링
# ========================================
def crawl_all_keywords(keywords_list, max_pages=2, max_workers=10):
    """병렬 크롤링 실행"""
    all_articles = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []

        for keyword in keywords_list:
            futures.append(executor.submit(crawl_naver_news_v2, keyword, max_pages))
            futures.append(executor.submit(crawl_daum_news_v2, keyword, max_pages))

        for future in tqdm(as_completed(futures), total=len(futures), desc="크롤링 진행"):
            try:
                result = future.result()
                all_articles.extend(result)
            except Exception as e:
                continue

    return all_articles

# ========================================
# 7단계: 메인 실행
# ========================================
def main(test_mode=False):
    print("="*70)
    print("🚀 증권 뉴스 크롤링 v2.0 (2025년 대응)")
    print("="*70)

    # 키워드 수집
    all_keywords = []

    for category, keywords in sector_trend_keywords.items():
        all_keywords.extend(keywords)

    for company, terms in company_keywords.items():
        all_keywords.extend(terms['company_terms'])
        all_keywords.extend(terms['game_terms'])

    print(f"📊 총 {len(all_keywords)}개 키워드")

    # 테스트 모드: 구조 확인
    if test_mode:
        print("\n🔍 테스트 모드: HTML 구조 확인")
        print("="*70)
        test_keywords = ["엔씨소프트", "리니지", "크래프톤"]
        for kw in test_keywords:
            test_naver_structure(kw)
            time.sleep(1)
        return None

    # 실제 크롤링
    print(f"🔍 샘플 키워드: {all_keywords[:5]}")
    print()

    start_time = time.time()
    articles = crawl_all_keywords(all_keywords, max_pages=2, max_workers=10)
    end_time = time.time()

    print()
    print("="*70)
    print(f"✅ 크롤링 완료!")
    print(f"⏱️  소요 시간: {end_time - start_time:.2f}초")
    print(f"📰 수집된 기사 수: {len(articles)}개")

    if articles:
        df = pd.DataFrame(articles)

        # 중복 제거
        original_count = len(df)
        df = df.drop_duplicates(subset=['link'], keep='first')
        print(f"🔄 중복 제거: {original_count}개 → {len(df)}개")

        # 키워드별 통계
        print("\n📊 키워드별 기사 수:")
        keyword_counts = df['keyword'].value_counts().head(10)
        for kw, cnt in keyword_counts.items():
            print(f"  - {kw}: {cnt}개")

        # CSV 저장
        filename = f"news_crawling_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 저장 완료: {filename}")

        # 미리보기
        print("\n📋 데이터 미리보기:")
        print(df[['source', 'keyword', 'title', 'press']].head(10))

        return df
    else:
        print("⚠️  수집된 기사가 없습니다.")
        print("\n💡 해결 방법:")
        print("1. test_mode=True로 실행하여 HTML 구조 확인")
        print("2. 네트워크 연결 확인")
        print("3. 키워드를 더 일반적인 단어로 변경")
        return None

# ========================================
# 실행
# ========================================

# 먼저 테스트 모드로 구조 확인
print("STEP 1: HTML 구조 테스트")
print("="*70)
main(test_mode=True)

print("\n\n")
print("STEP 2: 실제 크롤링 시작")
print("="*70)
time.sleep(2)

# 실제 크롤링
df_result = main(test_mode=False)

STEP 1: HTML 구조 테스트
🚀 증권 뉴스 크롤링 v2.0 (2025년 대응)
📊 총 62개 키워드

🔍 테스트 모드: HTML 구조 확인

키워드: 엔씨소프트
div.news_wrap: 0개 발견
div.news_area: 0개 발견
div.news_wrap.api_ani_send: 0개 발견
li.bx: 7개 발견
✅ 사용 가능한 선택자: li.bx

하위 구조:
  - 제목: None...
  - 설명: None

키워드: 리니지
div.news_wrap: 0개 발견
div.news_area: 0개 발견
div.news_wrap.api_ani_send: 0개 발견
li.bx: 7개 발견
✅ 사용 가능한 선택자: li.bx

하위 구조:
  - 제목: None...
  - 설명: None

키워드: 크래프톤
div.news_wrap: 0개 발견
div.news_area: 0개 발견
div.news_wrap.api_ani_send: 0개 발견
li.bx: 7개 발견
✅ 사용 가능한 선택자: li.bx

하위 구조:
  - 제목: None...
  - 설명: None



STEP 2: 실제 크롤링 시작
🚀 증권 뉴스 크롤링 v2.0 (2025년 대응)
📊 총 62개 키워드
🔍 샘플 키워드: ['신작 출시', '출시 임박', '흥행', '매출 1위', '사전예약']



크롤링 진행:   0%|          | 0/124 [00:00<?, ?it/s]

⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (신작 출시, page 1)
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (사전예약, page 1)
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (출시 임박, page 1)
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (매출 1위, page 1)
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (흥행, page 1)
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (신작 출시, page 2)
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (출시 임박, page 2)
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (매출 1위, page 2)
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (흥행, page 2)
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (사전예약, page 2)
⚠️ 네이버 응답 오류 (중국 판호, page 1): 403
⚠️ 네이버 응답 오류 (판호 발급, page 1): 403
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (중국 판호, page 1)
⚠️ 네이버 응답 오류 (글로벌 론칭, page 1): 403
⚠️ 네이버 응답 오류 (중국 판호, page 2): 403
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (판호 발급, page 1)
⚠️ 네이버 응답 오류 (판호 발급, page 2): 403
⚠️ 네이버 응답 오류 (글로벌 론칭, page 2): 403
⚠️ 네이버 응답 오류 (다운로드 1위, page 1): 403
⚠️ 네이버 응답 오류 (유저수 증가, page 1): 403
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (중국 판호, page 2)
⚠️ 네이버 응답 오류 (IP 확장, page 1): 403
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (글로벌 론칭, page 1)
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (다운로드 1위, page 1)
⚠️ 네이버 응답 오류 (다운로드 1위, page 2): 403
⚠️ 네이버 응답 오류 (콘솔 진출, page 1): 403
⚠️ 뉴스 항목을 찾을 수 없음 - 다음 (판호 발급, page 2)


In [ ]:
# ========================================
# 네이버+다음 뉴스 크롤링 v3.0 (li.bx 구조 분석)
# ========================================

!pip install requests beautifulsoup4 pandas tqdm lxml -q

import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import time
from urllib.parse import quote
import warnings
warnings.filterwarnings('ignore')

# ========================================
# 키워드 정의
# ========================================
sector_trend_keywords = {
    "호재": [
        "신작 출시", "출시 임박", "흥행", "매출 1위", "사전예약",
        "중국 판호", "판호 발급", "글로벌 론칭", "다운로드 1위", "유저수 증가",
        "IP 확장", "콘솔 진출", "멀티 플랫폼", "라이브 서비스",
        "인공지능 NPC", "AI 기술 적용", "그래픽 업그레이드"
    ],
    "악재": [
        "흥행 부진", "매출 감소", "유저 이탈", "과금 논란", "확률형 아이템 논란",
        "규제 강화", "판호 불허", "심의 보류",
        "점검 논란", "서버 장애", "오류 발생",
        "리뷰 폭격", "평점 하락", "환불 사태"
    ]
}

company_keywords = {
    "NCSoft": {
        "company_terms": ["엔씨소프트", "NC소프트", "NCSoft"],
        "game_terms": [
            "리니지", "리니지M", "리니지W", "리니지2M",
            "아이온", "아이온2", "블레이드앤소울", "블레이드 & 소울",
            "프로젝트 LLL", "LLL"
        ],
    },
    "Krafton": {
        "company_terms": ["크래프톤", "KRAFTON"],
        "game_terms": [
            "배틀그라운드", "배그", "PUBG", "PUBG MOBILE",
            "다크앤다커", "Dark and Darker", "Dark and Darker Mobile",
            "inZOI", "인조이"
        ],
    },
    "Mgame": {
        "company_terms": ["엠게임", "Mgame"],
        "game_terms": [
            "열혈강호", "열혈강호 온라인", "열혈강호M",
            "나이트 온라인", "Knight Online"
        ],
    },
}

# ========================================
# 상세 HTML 구조 분석
# ========================================
def analyze_html_structure(keyword):
    """li.bx 내부 구조 상세 분석"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    url = f"https://search.naver.com/search.naver?where=news&query={quote(keyword)}&sort=1"

    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'lxml')

        items = soup.select('li.bx')

        if items:
            print(f"\n키워드: {keyword}")
            print("="*80)
            print(f"li.bx 항목 수: {len(items)}개\n")

            # 첫 번째 항목 상세 분석
            first_item = items[0]
            print("첫 번째 항목의 HTML 구조:")
            print("-"*80)
            print(first_item.prettify()[:2000])
            print("-"*80)

            # 가능한 모든 a 태그 찾기
            print("\n발견된 모든 <a> 태그:")
            all_links = first_item.find_all('a')
            for i, link in enumerate(all_links):
                print(f"{i+1}. class={link.get('class')}, href={link.get('href')[:50] if link.get('href') else 'None'}")
                print(f"   텍스트: {link.get_text(strip=True)[:50]}")

            # 클래스별 검색
            print("\n\n주요 클래스 검색 결과:")
            test_selectors = [
                'a.news_tit', 'a.tit', 'a',
                'div.news_area', 'div.news_contents',
                'div.dsc', 'div.news_dsc',
                'span.press', 'a.press'
            ]

            for sel in test_selectors:
                found = first_item.select(sel)
                if found:
                    print(f"✅ {sel}: {len(found)}개")
                    if found[0].get_text(strip=True):
                        print(f"   샘플: {found[0].get_text(strip=True)[:60]}")

            return first_item
        else:
            print(f"⚠️ {keyword}: li.bx를 찾을 수 없습니다.")
            return None

    except Exception as e:
        print(f"오류: {e}")
        return None

# ========================================
# 개선된 네이버 크롤링 (li.bx 기반)
# ========================================
def crawl_naver_news_v3(keyword, max_pages=3):
    """li.bx 구조에 최적화된 크롤링"""
    articles = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': 'https://www.naver.com/'
    }

    for page in range(1, max_pages + 1):
        try:
            start = 1 + (page - 1) * 10
            url = f"https://search.naver.com/search.naver?where=news&query={quote(keyword)}&sort=1&start={start}"

            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code != 200:
                continue

            soup = BeautifulSoup(response.text, 'lxml')
            news_items = soup.select('li.bx')

            if not news_items:
                continue

            for item in news_items:
                try:
                    # 모든 a 태그에서 뉴스 링크 찾기
                    all_links = item.find_all('a')

                    title = None
                    link = None

                    # 네이버 뉴스 링크 찾기 (news.naver.com 포함)
                    for a_tag in all_links:
                        href = a_tag.get('href', '')
                        text = a_tag.get_text(strip=True)

                        # 뉴스 제목은 보통 가장 긴 텍스트
                        if text and len(text) > 10 and 'news.naver.com' in href:
                            if title is None or len(text) > len(title):
                                title = text
                                link = href

                    # 제목을 못 찾았다면 첫 번째 링크 사용
                    if not title and all_links:
                        title = all_links[0].get_text(strip=True)
                        link = all_links[0].get('href', '')

                    if not title:
                        continue

                    # 설명 - 여러 가능성 시도
                    description = ""
                    desc_selectors = ['div.dsc', 'dd', 'div.news_dsc', 'p.dsc']
                    for sel in desc_selectors:
                        desc_elem = item.select_one(sel)
                        if desc_elem:
                            description = desc_elem.get_text(strip=True)
                            break

                    # 언론사
                    press = ""
                    press_selectors = ['span.press', 'a.press', 'cite']
                    for sel in press_selectors:
                        press_elem = item.select_one(sel)
                        if press_elem:
                            press = press_elem.get_text(strip=True)
                            break

                    # 날짜
                    date = ""
                    date_selectors = ['dd.txt_inline', 'span.date', 'span']
                    for sel in date_selectors:
                        date_elem = item.select_one(sel)
                        if date_elem:
                            date_text = date_elem.get_text(strip=True)
                            # 날짜 형식인지 확인 (숫자 포함)
                            if any(char.isdigit() for char in date_text):
                                date = date_text
                                break

                    articles.append({
                        'source': '네이버',
                        'keyword': keyword,
                        'title': title,
                        'description': description,
                        'press': press,
                        'date': date,
                        'link': link,
                        'crawled_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    })

                except Exception as e:
                    continue

            time.sleep(0.5)

        except Exception as e:
            continue

    return articles

# ========================================
# 다음 뉴스 크롤링
# ========================================
def crawl_daum_news_v3(keyword, max_pages=3):
    """다음 뉴스 크롤링"""
    articles = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': 'https://www.daum.net/'
    }

    for page in range(1, max_pages + 1):
        try:
            url = f"https://search.daum.net/search?w=news&q={quote(keyword)}&sort=recency&p={page}"

            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code != 200:
                continue

            soup = BeautifulSoup(response.text, 'lxml')

            # 다음 뉴스는 여러 구조 가능
            news_items = soup.select('c-doc') or soup.select('div.news_wrap')

            if not news_items:
                continue

            for item in news_items:
                try:
                    # 제목과 링크
                    title_elem = item.select_one('a.tit_main') or item.select_one('a[class*="tit"]') or item.select_one('strong.tit_g')

                    if not title_elem:
                        # 모든 a 태그 시도
                        all_links = item.find_all('a')
                        for link in all_links:
                            if len(link.get_text(strip=True)) > 10:
                                title_elem = link
                                break

                    if not title_elem:
                        continue

                    title = title_elem.get_text(strip=True)
                    link = title_elem.get('href', '')

                    # 설명
                    desc_elem = item.select_one('p.desc') or item.select_one('p[class*="desc"]')
                    description = desc_elem.get_text(strip=True) if desc_elem else ""

                    # 언론사
                    press_elem = item.select_one('span.txt_info') or item.select_one('span[class*="cp"]')
                    press = press_elem.get_text(strip=True) if press_elem else ""

                    # 날짜
                    date_elem = item.select_one('span.gem_txt') or item.select_one('span[class*="date"]')
                    date = date_elem.get_text(strip=True) if date_elem else ""

                    articles.append({
                        'source': '다음',
                        'keyword': keyword,
                        'title': title,
                        'description': description,
                        'press': press,
                        'date': date,
                        'link': link,
                        'crawled_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    })

                except Exception as e:
                    continue

            time.sleep(0.5)

        except Exception as e:
            continue

    return articles

# ========================================
# 병렬 크롤링
# ========================================
def crawl_all_keywords(keywords_list, max_pages=2, max_workers=10):
    """병렬 크롤링"""
    all_articles = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []

        for keyword in keywords_list:
            futures.append(executor.submit(crawl_naver_news_v3, keyword, max_pages))
            futures.append(executor.submit(crawl_daum_news_v3, keyword, max_pages))

        for future in tqdm(as_completed(futures), total=len(futures), desc="크롤링 진행"):
            try:
                result = future.result()
                all_articles.extend(result)
            except Exception as e:
                continue

    return all_articles

# ========================================
# 메인 실행
# ========================================
def main(test_mode=False, analyze_mode=False):
    print("="*80)
    print("🚀 증권 뉴스 크롤링 v3.0 (li.bx 구조 최적화)")
    print("="*80)

    # 키워드 수집
    all_keywords = []

    for category, keywords in sector_trend_keywords.items():
        all_keywords.extend(keywords)

    for company, terms in company_keywords.items():
        all_keywords.extend(terms['company_terms'])
        all_keywords.extend(terms['game_terms'])

    print(f"📊 총 {len(all_keywords)}개 키워드")

    # 상세 분석 모드
    if analyze_mode:
        print("\n🔍 상세 분석 모드: li.bx 내부 구조 확인")
        print("="*80)
        test_keywords = ["엔씨소프트", "리니지"]
        for kw in test_keywords:
            analyze_html_structure(kw)
            print("\n" + "="*80 + "\n")
            time.sleep(1)
        return None

    # 테스트 모드
    if test_mode:
        print("\n🧪 테스트 모드: 소량 크롤링")
        print("="*80)
        test_keywords = ["엔씨소프트", "리니지", "크래프톤"]

        for kw in test_keywords:
            print(f"\n테스트 키워드: {kw}")
            naver_articles = crawl_naver_news_v3(kw, max_pages=1)
            daum_articles = crawl_daum_news_v3(kw, max_pages=1)

            print(f"  네이버: {len(naver_articles)}개")
            print(f"  다음: {len(daum_articles)}개")

            if naver_articles:
                print(f"  샘플: {naver_articles[0]['title'][:50]}")

        return None

    # 실제 크롤링
    print(f"\n🔍 샘플 키워드: {all_keywords[:5]}")
    print()

    start_time = time.time()
    articles = crawl_all_keywords(all_keywords, max_pages=2, max_workers=10)
    end_time = time.time()

    print()
    print("="*80)
    print(f"✅ 크롤링 완료!")
    print(f"⏱️  소요 시간: {end_time - start_time:.2f}초")
    print(f"📰 수집된 기사 수: {len(articles)}개")

    if articles:
        df = pd.DataFrame(articles)

        # 중복 제거
        original_count = len(df)
        df = df.drop_duplicates(subset=['link'], keep='first')
        print(f"🔄 중복 제거: {original_count}개 → {len(df)}개")

        # 소스별 통계
        print(f"\n📊 소스별 통계:")
        print(df['source'].value_counts())

        # 키워드별 통계 (상위 10개)
        print(f"\n📊 키워드별 기사 수 (상위 10개):")
        keyword_counts = df['keyword'].value_counts().head(10)
        for kw, cnt in keyword_counts.items():
            print(f"  - {kw}: {cnt}개")

        # CSV 저장
        filename = f"news_crawling_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 저장 완료: {filename}")

        # 미리보기
        print("\n📋 데이터 미리보기:")
        print(df[['source', 'keyword', 'title', 'press', 'date']].head(10))

        return df
    else:
        print("\n⚠️  수집된 기사가 없습니다.")
        print("\n💡 다음 단계:")
        print("1. main(analyze_mode=True) 실행으로 HTML 구조 상세 분석")
        print("2. main(test_mode=True) 실행으로 소량 테스트")
        return None

# ========================================
# 실행
# ========================================

# STEP 1: 상세 구조 분석
print("STEP 1: li.bx 내부 HTML 구조 상세 분석")
print("="*80)
main(analyze_mode=True)

print("\n\n")

# STEP 2: 소량 테스트
print("STEP 2: 소량 테스트 크롤링")
print("="*80)
main(test_mode=True)

print("\n\n")

# STEP 3: 실제 크롤링 (위 테스트가 성공하면 실행)
# print("STEP 3: 전체 크롤링")
# print("="*80)
# df_result = main(test_mode=False)

STEP 1: li.bx 내부 HTML 구조 상세 분석
🚀 증권 뉴스 크롤링 v3.0 (li.bx 구조 최적화)
📊 총 62개 키워드

🔍 상세 분석 모드: li.bx 내부 구조 확인

키워드: 엔씨소프트
li.bx 항목 수: 7개

첫 번째 항목의 HTML 구조:
--------------------------------------------------------------------------------
<li class="bx lineup">
 <div class="bx_inner">
  <strong class="tit">
   정렬
  </strong>
  <div class="option" role="tablist">
   <a aria-selected="false" class="txt" href="#" onclick='return news_submit_sort_option(0,"sim"),!1' role="tab">
    관련도순
   </a>
   <a aria-selected="true" class="txt" href="#" onclick='return news_submit_sort_option(1,"new"),!1' role="tab">
    최신순
   </a>
   <a aria-selected="false" class="txt" href="#" onclick='return news_submit_sort_option(2,"old"),!1' role="tab">
    오래된순
   </a>
  </div>
 </div>
</li>

--------------------------------------------------------------------------------

발견된 모든 <a> 태그:
1. class=['txt'], href=#
   텍스트: 관련도순
2. class=['txt'], href=#
   텍스트: 최신순
3. class=['txt'], href=#
   텍스트: 오래된순


주요 클래스 검색 결과:
✅ a

In [ ]:
# ========================================
# 네이버+다음 뉴스 크롤링 v3.0 (li.bx 구조 분석)
# ========================================

!pip install requests beautifulsoup4 pandas tqdm lxml -q

import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import time
from urllib.parse import quote
import warnings
warnings.filterwarnings('ignore')

# ========================================
# 키워드 정의
# ========================================
sector_trend_keywords = {
    "호재": [
        "신작 출시", "출시 임박", "흥행", "매출 1위", "사전예약",
        "중국 판호", "판호 발급", "글로벌 론칭", "다운로드 1위", "유저수 증가",
        "IP 확장", "콘솔 진출", "멀티 플랫폼", "라이브 서비스",
        "인공지능 NPC", "AI 기술 적용", "그래픽 업그레이드"
    ],
    "악재": [
        "흥행 부진", "매출 감소", "유저 이탈", "과금 논란", "확률형 아이템 논란",
        "규제 강화", "판호 불허", "심의 보류",
        "점검 논란", "서버 장애", "오류 발생",
        "리뷰 폭격", "평점 하락", "환불 사태"
    ]
}

company_keywords = {
    "NCSoft": {
        "company_terms": ["엔씨소프트", "NC소프트", "NCSoft"],
        "game_terms": [
            "리니지", "리니지M", "리니지W", "리니지2M",
            "아이온", "아이온2", "블레이드앤소울", "블레이드 & 소울",
            "프로젝트 LLL", "LLL"
        ],
    },
    "Krafton": {
        "company_terms": ["크래프톤", "KRAFTON"],
        "game_terms": [
            "배틀그라운드", "배그", "PUBG", "PUBG MOBILE",
            "다크앤다커", "Dark and Darker", "Dark and Darker Mobile",
            "inZOI", "인조이"
        ],
    },
    "Mgame": {
        "company_terms": ["엠게임", "Mgame"],
        "game_terms": [
            "열혈강호", "열혈강호 온라인", "열혈강호M",
            "나이트 온라인", "Knight Online"
        ],
    },
}

# ========================================
# 상세 HTML 구조 분석
# ========================================
def analyze_html_structure(keyword):
    """li.bx 내부 구조 상세 분석"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    url = f"https://search.naver.com/search.naver?where=news&query={quote(keyword)}&sort=1"

    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'lxml')

        items = soup.select('li.bx')

        if items:
            print(f"\n키워드: {keyword}")
            print("="*80)
            print(f"li.bx 항목 수: {len(items)}개\n")

            # 모든 항목 확인 (첫 3개)
            for idx, item in enumerate(items[:3]):
                print(f"\n{'='*80}")
                print(f"항목 #{idx+1}:")
                print(f"{'='*80}")
                print(item.prettify()[:1500])
                print("...")

            # 뉴스 항목 찾기 (정렬 옵션 제외)
            print(f"\n\n{'='*80}")
            print("뉴스 항목 분석 (정렬 옵션 제외):")
            print(f"{'='*80}")

            for idx, item in enumerate(items):
                # 정렬 옵션인지 확인
                if 'lineup' in item.get('class', []):
                    print(f"항목 #{idx+1}: 정렬 옵션 (건너뜀)")
                    continue

                print(f"\n항목 #{idx+1}: 뉴스 항목")
                print("-"*80)

                # 가능한 모든 a 태그 찾기
                all_links = item.find_all('a')
                print(f"발견된 <a> 태그 수: {len(all_links)}개")

                for i, link in enumerate(all_links[:3]):
                    href = link.get('href', '')
                    text = link.get_text(strip=True)
                    classes = link.get('class', [])
                    print(f"  {i+1}. class={classes}")
                    print(f"     href={href[:60]}")
                    print(f"     text={text[:60]}")

                # HTML 구조 일부
                print("\nHTML 구조:")
                print(item.prettify()[:1000])
                print("...")

                break  # 첫 번째 실제 뉴스만 분석

            # 클래스별 검색
            print("\n\n주요 선택자 테스트:")
            test_selectors = [
                'a.news_tit', 'a.tit', 'div.news_area',
                'div.news_contents', 'div.dsc', 'span.info'
            ]

            for item in items:
                if 'lineup' in item.get('class', []):
                    continue

                for sel in test_selectors:
                    found = item.select(sel)
                    if found:
                        print(f"✅ {sel}: {len(found)}개")
                        if found[0].get_text(strip=True):
                            print(f"   샘플: {found[0].get_text(strip=True)[:60]}")

                break  # 첫 번째 실제 뉴스만

            return first_item
        else:
            print(f"⚠️ {keyword}: li.bx를 찾을 수 없습니다.")
            return None

    except Exception as e:
        print(f"오류: {e}")
        return None

# ========================================
# 개선된 네이버 크롤링 (li.bx 기반)
# ========================================
def crawl_naver_news_v3(keyword, max_pages=3):
    """li.bx 구조에 최적화된 크롤링"""
    articles = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': 'https://www.naver.com/'
    }

    for page in range(1, max_pages + 1):
        try:
            start = 1 + (page - 1) * 10
            url = f"https://search.naver.com/search.naver?where=news&query={quote(keyword)}&sort=1&start={start}"

            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code != 200:
                continue

            soup = BeautifulSoup(response.text, 'lxml')

            # 정렬 옵션 제외하고 뉴스 항목만 선택
            all_items = soup.select('li.bx')
            news_items = [item for item in all_items if 'lineup' not in item.get('class', [])]

            if not news_items:
                continue

            for item in news_items:
                try:
                    # 모든 a 태그에서 뉴스 링크 찾기
                    all_links = item.find_all('a')

                    title = None
                    link = None

                    # 네이버 뉴스 링크 찾기 (news.naver.com 포함)
                    for a_tag in all_links:
                        href = a_tag.get('href', '')
                        text = a_tag.get_text(strip=True)

                        # 뉴스 제목은 보통 가장 긴 텍스트
                        if text and len(text) > 10 and 'news.naver.com' in href:
                            if title is None or len(text) > len(title):
                                title = text
                                link = href

                    # 제목을 못 찾았다면 첫 번째 링크 사용
                    if not title and all_links:
                        title = all_links[0].get_text(strip=True)
                        link = all_links[0].get('href', '')

                    if not title:
                        continue

                    # 설명 - 여러 가능성 시도
                    description = ""
                    desc_selectors = ['div.dsc', 'dd', 'div.news_dsc', 'p.dsc']
                    for sel in desc_selectors:
                        desc_elem = item.select_one(sel)
                        if desc_elem:
                            description = desc_elem.get_text(strip=True)
                            break

                    # 언론사
                    press = ""
                    press_selectors = ['span.press', 'a.press', 'cite']
                    for sel in press_selectors:
                        press_elem = item.select_one(sel)
                        if press_elem:
                            press = press_elem.get_text(strip=True)
                            break

                    # 날짜
                    date = ""
                    date_selectors = ['dd.txt_inline', 'span.date', 'span']
                    for sel in date_selectors:
                        date_elem = item.select_one(sel)
                        if date_elem:
                            date_text = date_elem.get_text(strip=True)
                            # 날짜 형식인지 확인 (숫자 포함)
                            if any(char.isdigit() for char in date_text):
                                date = date_text
                                break

                    articles.append({
                        'source': '네이버',
                        'keyword': keyword,
                        'title': title,
                        'description': description,
                        'press': press,
                        'date': date,
                        'link': link,
                        'crawled_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    })

                except Exception as e:
                    continue

            time.sleep(0.5)

        except Exception as e:
            continue

    return articles

# ========================================
# 다음 뉴스 크롤링
# ========================================
def crawl_daum_news_v3(keyword, max_pages=3):
    """다음 뉴스 크롤링"""
    articles = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': 'https://www.daum.net/'
    }

    for page in range(1, max_pages + 1):
        try:
            url = f"https://search.daum.net/search?w=news&q={quote(keyword)}&sort=recency&p={page}"

            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code != 200:
                continue

            soup = BeautifulSoup(response.text, 'lxml')

            # 다음 뉴스는 여러 구조 가능
            news_items = soup.select('c-doc') or soup.select('div.news_wrap')

            if not news_items:
                continue

            for item in news_items:
                try:
                    # 제목과 링크
                    title_elem = item.select_one('a.tit_main') or item.select_one('a[class*="tit"]') or item.select_one('strong.tit_g')

                    if not title_elem:
                        # 모든 a 태그 시도
                        all_links = item.find_all('a')
                        for link in all_links:
                            if len(link.get_text(strip=True)) > 10:
                                title_elem = link
                                break

                    if not title_elem:
                        continue

                    title = title_elem.get_text(strip=True)
                    link = title_elem.get('href', '')

                    # 설명
                    desc_elem = item.select_one('p.desc') or item.select_one('p[class*="desc"]')
                    description = desc_elem.get_text(strip=True) if desc_elem else ""

                    # 언론사
                    press_elem = item.select_one('span.txt_info') or item.select_one('span[class*="cp"]')
                    press = press_elem.get_text(strip=True) if press_elem else ""

                    # 날짜
                    date_elem = item.select_one('span.gem_txt') or item.select_one('span[class*="date"]')
                    date = date_elem.get_text(strip=True) if date_elem else ""

                    articles.append({
                        'source': '다음',
                        'keyword': keyword,
                        'title': title,
                        'description': description,
                        'press': press,
                        'date': date,
                        'link': link,
                        'crawled_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    })

                except Exception as e:
                    continue

            time.sleep(0.5)

        except Exception as e:
            continue

    return articles

# ========================================
# 병렬 크롤링
# ========================================
def crawl_all_keywords(keywords_list, max_pages=2, max_workers=10):
    """병렬 크롤링"""
    all_articles = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = []

        for keyword in keywords_list:
            futures.append(executor.submit(crawl_naver_news_v3, keyword, max_pages))
            futures.append(executor.submit(crawl_daum_news_v3, keyword, max_pages))

        for future in tqdm(as_completed(futures), total=len(futures), desc="크롤링 진행"):
            try:
                result = future.result()
                all_articles.extend(result)
            except Exception as e:
                continue

    return all_articles

# ========================================
# 메인 실행
# ========================================
def main(test_mode=False, analyze_mode=False):
    print("="*80)
    print("🚀 증권 뉴스 크롤링 v3.0 (li.bx 구조 최적화)")
    print("="*80)

    # 키워드 수집
    all_keywords = []

    for category, keywords in sector_trend_keywords.items():
        all_keywords.extend(keywords)

    for company, terms in company_keywords.items():
        all_keywords.extend(terms['company_terms'])
        all_keywords.extend(terms['game_terms'])

    print(f"📊 총 {len(all_keywords)}개 키워드")

    # 상세 분석 모드
    if analyze_mode:
        print("\n🔍 상세 분석 모드: li.bx 내부 구조 확인")
        print("="*80)
        test_keywords = ["엔씨소프트", "리니지"]
        for kw in test_keywords:
            analyze_html_structure(kw)
            print("\n" + "="*80 + "\n")
            time.sleep(1)
        return None

    # 테스트 모드
    if test_mode:
        print("\n🧪 테스트 모드: 소량 크롤링")
        print("="*80)
        test_keywords = ["엔씨소프트", "리니지", "크래프톤"]

        for kw in test_keywords:
            print(f"\n테스트 키워드: {kw}")
            naver_articles = crawl_naver_news_v3(kw, max_pages=1)
            daum_articles = crawl_daum_news_v3(kw, max_pages=1)

            print(f"  네이버: {len(naver_articles)}개")
            print(f"  다음: {len(daum_articles)}개")

            if naver_articles:
                print(f"  샘플: {naver_articles[0]['title'][:50]}")

        return None

    # 실제 크롤링
    print(f"\n🔍 샘플 키워드: {all_keywords[:5]}")
    print()

    start_time = time.time()
    articles = crawl_all_keywords(all_keywords, max_pages=2, max_workers=10)
    end_time = time.time()

    print()
    print("="*80)
    print(f"✅ 크롤링 완료!")
    print(f"⏱️  소요 시간: {end_time - start_time:.2f}초")
    print(f"📰 수집된 기사 수: {len(articles)}개")

    if articles:
        df = pd.DataFrame(articles)

        # 중복 제거
        original_count = len(df)
        df = df.drop_duplicates(subset=['link'], keep='first')
        print(f"🔄 중복 제거: {original_count}개 → {len(df)}개")

        # 소스별 통계
        print(f"\n📊 소스별 통계:")
        print(df['source'].value_counts())

        # 키워드별 통계 (상위 10개)
        print(f"\n📊 키워드별 기사 수 (상위 10개):")
        keyword_counts = df['keyword'].value_counts().head(10)
        for kw, cnt in keyword_counts.items():
            print(f"  - {kw}: {cnt}개")

        # CSV 저장
        filename = f"news_crawling_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"\n💾 저장 완료: {filename}")

        # 미리보기
        print("\n📋 데이터 미리보기:")
        print(df[['source', 'keyword', 'title', 'press', 'date']].head(10))

        return df
    else:
        print("\n⚠️  수집된 기사가 없습니다.")
        print("\n💡 다음 단계:")
        print("1. main(analyze_mode=True) 실행으로 HTML 구조 상세 분석")
        print("2. main(test_mode=True) 실행으로 소량 테스트")
        return None

# ========================================
# 실행
# ========================================

# STEP 1: 상세 구조 분석
print("STEP 1: li.bx 내부 HTML 구조 상세 분석")
print("="*80)
main(analyze_mode=True)

print("\n\n")

# STEP 2: 소량 테스트
print("STEP 2: 소량 테스트 크롤링")
print("="*80)
main(test_mode=True)

print("\n\n")

# STEP 3: 실제 크롤링 (위 테스트가 성공하면 실행)
# print("STEP 3: 전체 크롤링")
# print("="*80)
# df_result = main(test_mode=False)

STEP 1: li.bx 내부 HTML 구조 상세 분석
🚀 증권 뉴스 크롤링 v3.0 (li.bx 구조 최적화)
📊 총 62개 키워드

🔍 상세 분석 모드: li.bx 내부 구조 확인

키워드: 엔씨소프트
li.bx 항목 수: 7개


항목 #1:
<li class="bx lineup">
 <div class="bx_inner">
  <strong class="tit">
   정렬
  </strong>
  <div class="option" role="tablist">
   <a aria-selected="false" class="txt" href="#" onclick='return news_submit_sort_option(0,"sim"),!1' role="tab">
    관련도순
   </a>
   <a aria-selected="true" class="txt" href="#" onclick='return news_submit_sort_option(1,"new"),!1' role="tab">
    최신순
   </a>
   <a aria-selected="false" class="txt" href="#" onclick='return news_submit_sort_option(2,"old"),!1' role="tab">
    오래된순
   </a>
  </div>
 </div>
</li>

...

항목 #2:
<li class="bx service">
 <div class="bx_inner">
  <strong class="tit">
   서비스 영역
  </strong>
  <div class="option" role="tablist">
   <a aria-selected="true" class="txt" data-search-option-item="" href="javascript:;" onclick='return news_submit_service_option(0,"serviceall"),!1' role="tab">
    전체
   </a>
 

In [ ]:
import requests
from bs4 import BeautifulSoup

# Define your keywords
sector_trend_keywords = {
    "호재": [
        "신작 출시", "출시 임박", "흥행", "매출 1위", "사전예약",
        "중국 판호", "판호 발급", "글로벌 론칭", "다운로드 1위", "유저수 증가",
        "IP 확장", "콘솔 진출", "멀티 플랫폼", "라이브 서비스",
        "인공지능 NPC", "AI 기술 적용", "그래픽 업그레이드"
    ],
    "악재": [
        "흥행 부진", "매출 감소", "유저 이탈", "과금 논란", "확률형 아이템 논란",
        "규제 강화", "판호 불허", "심의 보류",
        "점검 논란", "서버 장애", "오류 발생",
        "리뷰 폭격", "평점 하락", "환불 사태"
    ]
}

company_keywords = {
    "NCSoft": {
        "company_terms": ["엔씨소프트", "NC소프트", "NCSoft"],
        "game_terms": [
            "리니지", "리니지M", "리니지W", "리니지2M",
            "아이온", "아이온2", "블레이드앤소울", "블레이드 & 소울",
            "프로젝트 LLL", "LLL"
        ],
    },
    "Krafton": {
        "company_terms": ["크래프톤", "KRAFTON"],
        "game_terms": [
            "배틀그라운드", "배그", "PUBG", "PUBG MOBILE",
            "다크앤다커", "Dark and Darker", "Dark and Darker Mobile",
            "inZOI", "인조이"
        ],
    },
    "Mgame": {
        "company_terms": ["엠게임", "Mgame"],
        "game_terms": [
            "열혈강호", "열혈강호 온라인", "열혈강호M",
            "나이트 온라인", "Knight Online"
        ],
    },
}

def crawl_naver(keyword):
    """Crawl Naver for the specified keyword and return titles and links."""
    url = f"https://search.naver.com/search.naver?query={keyword}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36"
    }
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        # Extract relevant data from the page
        results = []
        for item in soup.select('.news_area .news_tit'):
            title = item.get_text(strip=True)
            link = item['href']
            results.append((title, link))
        return results
    else:
        print(f"Failed to retrieve data for keyword: {keyword}")
        return []

# Main function to iterate through keywords
def main():
    # Crawl for sector trend keywords
    for sector, keywords in sector_trend_keywords.items():
        for keyword in keywords:
            print(f"Crawling Naver for sector '{sector}' with keyword '{keyword}'")
            results = crawl_naver(keyword)
            for title, link in results:
                print(f"Title: {title}, Link: {link}")

    # Crawl for company keywords
    for company, data in company_keywords.items():
        for term in data['company_terms'] + data['game_terms']:
            print(f"Crawling Naver for company '{company}' with term '{term}'")
            results = crawl_naver(term)
            for title, link in results:
                print(f"Title: {title}, Link: {link}")

if __name__ == "__main__":
    main()


Crawling Naver for sector '호재' with keyword '신작 출시'
Crawling Naver for sector '호재' with keyword '출시 임박'
Crawling Naver for sector '호재' with keyword '흥행'
Crawling Naver for sector '호재' with keyword '매출 1위'
Crawling Naver for sector '호재' with keyword '사전예약'
Crawling Naver for sector '호재' with keyword '중국 판호'
Crawling Naver for sector '호재' with keyword '판호 발급'
Crawling Naver for sector '호재' with keyword '글로벌 론칭'
Crawling Naver for sector '호재' with keyword '다운로드 1위'
Crawling Naver for sector '호재' with keyword '유저수 증가'
Crawling Naver for sector '호재' with keyword 'IP 확장'
Crawling Naver for sector '호재' with keyword '콘솔 진출'
Crawling Naver for sector '호재' with keyword '멀티 플랫폼'
Crawling Naver for sector '호재' with keyword '라이브 서비스'
Crawling Naver for sector '호재' with keyword '인공지능 NPC'
Crawling Naver for sector '호재' with keyword 'AI 기술 적용'
Crawling Naver for sector '호재' with keyword '그래픽 업그레이드'
Crawling Naver for sector '악재' with keyword '흥행 부진'
Crawling Naver for sector '악재' with keyword '매출 감소'

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta, date
import time

# 1) 섹터 트렌드 키워드
sector_trend_keywords = {
    "호재": [
        "신작 출시", "출시 임박", "흥행", "매출 1위", "사전예약",
        "중국 판호", "판호 발급", "글로벌 론칭", "다운로드 1위", "유저수 증가",
        "IP 확장", "콘솔 진출", "멀티 플랫폼", "라이브 서비스",
        "인공지능 NPC", "AI 기술 적용", "그래픽 업그레이드"
    ],
    "악재": [
        "흥행 부진", "매출 감소", "유저 이탈", "과금 논란", "확률형 아이템 논란",
        "규제 강화", "판호 불허", "심의 보류",
        "점검 논란", "서버 장애", "오류 발생",
        "리뷰 폭격", "평점 하락", "환불 사태"
    ]
}

# 2) 회사별 키워드
company_keywords = {
    "NCSoft": {
        "company_terms": ["엔씨소프트", "NC소프트", "NCSoft"],
        "game_terms": [
            "리니지", "리니지M", "리니지W", "리니지2M",
            "아이온", "아이온2", "블레이드앤소울", "블레이드 & 소울",
            "프로젝트 LLL", "LLL"
        ],
    },
    "Krafton": {
        "company_terms": ["크래프톤", "KRAFTON"],
        "game_terms": [
            "배틀그라운드", "배그", "PUBG", "PUBG MOBILE",
            "다크앤다커", "Dark and Darker", "Dark and Darker Mobile",
            "inZOI", "인조이"
        ],
    },
    "Mgame": {
        "company_terms": ["엠게임", "Mgame"],
        "game_terms": [
            "열혈강호", "열혈강호 온라인", "열혈강호M",
            "나이트 온라인", "Knight Online"
        ],
    },
}


In [ ]:
def parse_naver_news_date(date_str: str) -> date:
    """
    네이버 뉴스 검색 결과의 날짜 텍스트를 date로 변환.
    예) '2025.01.10.', '3시간 전', '2일 전' 등
    """
    if not isinstance(date_str, str):
        date_str = str(date_str)
    date_str = date_str.strip()
    now = datetime.now()

    # 1) '2025.01.10.' 또는 '2025.01.10. 오전 9:30'
    try:
        base = date_str.split()[0].rstrip(".")  # '2025.01.10'
        dt = datetime.strptime(base, "%Y.%m.%d")
        return dt.date()
    except Exception:
        pass

    # 2) 'n분 전', 'n시간 전', 'n일 전'
    try:
        if "분 전" in date_str:
            mins = int(date_str.replace("분 전", "").strip())
            return (now - timedelta(minutes=mins)).date()
        if "시간 전" in date_str:
            hrs = int(date_str.replace("시간 전", "").strip())
            return (now - timedelta(hours=hrs)).date()
        if "일 전" in date_str:
            days = int(date_str.replace("일 전", "").strip())
            return (now - timedelta(days=days)).date()
    except Exception:
        pass

    # 3) 실패 시 오늘로
    return now.date()


In [ ]:
def crawl_naver_news(keyword: str,
                     days: int = 30,
                     max_pages: int = 3):
    """
    네이버 검색 > 뉴스 탭에서 keyword로 기사 크롤링.
    - where=news 사용
    - 레이아웃이 바뀌어도 a.news_tit 기준으로 최대한 긁어옴
    """
    base_url = "https://search.naver.com/search.naver"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    results = []

    for page in range(1, max_pages + 1):
        start = (page - 1) * 10 + 1  # 1, 11, 21...

        params = {
            "where": "news",
            "query": keyword,
            "start": start,
            "sort": 0,  # 0: 관련도, 1: 최신순
            "pd": 3,    # 3: 기간 직접 설정
            "ds": start_date.strftime("%Y.%m.%d"),
            "de": end_date.strftime("%Y.%m.%d"),
        }

        print(f"[{keyword}] 페이지 {page} 요청 중...")
        resp = requests.get(base_url, headers=headers, params=params, timeout=10)

        if resp.status_code != 200:
            print(f"  ❌ status_code={resp.status_code}")
            break

        from bs4 import BeautifulSoup
        soup = BeautifulSoup(resp.text, "html.parser")

        # 1차: div.news_area 안의 a.news_tit
        article_blocks = soup.select("div.news_area")
        anchors = soup.select("a.news_tit")

        if not anchors:
            # 완전 구조가 다를 때: 디버깅을 위해 상위 HTML 일부 출력
            print("  ⚠️ a.news_tit를 찾지 못했습니다. HTML 일부를 출력합니다.")
            print(soup.prettify()[:800])
            break

        print(f"  → a.news_tit {len(anchors)}개 발견")

        for a in anchors:
            try:
                title = a.get_text(strip=True)
                link = a["href"]

                # 가능한 경우, 부모 news_area 블록을 찾음
                news_area = a.find_parent("div", class_="news_area")
                if news_area:
                    summary_tag = (
                        news_area.select_one("div.news_dsc")
                        or news_area.select_one("div.dsc_wrap")
                    )
                    summary = summary_tag.get_text(strip=True) if summary_tag else ""

                    press_tag = (
                        news_area.select_one("a.info.press")
                        or news_area.select_one("span.press")
                        or news_area.select_one("span[class*='press']")
                    )
                    press = press_tag.get_text(strip=True) if press_tag else ""

                    date_tag = (
                        news_area.select_one("span.info")
                        or news_area.select_one("span[class*='date']")
                    )
                    date_str = date_tag.get_text(strip=True) if date_tag else ""
                else:
                    # news_area를 못 찾으면 최소한 제목/링크만 사용
                    summary = ""
                    press = ""
                    date_str = ""

                pub_date = parse_naver_news_date(date_str)

                results.append({
                    "keyword": keyword,
                    "title": title,
                    "summary": summary,
                    "link": link,
                    "press": press,
                    "date": pub_date,
                    "date_str": date_str,
                    "crawled_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                })

            except Exception as e:
                print(f"  ⚠️ 기사 파싱 오류: {e}")
                continue

        time.sleep(2)

    return results


In [ ]:
def run_full_crawling(
    sector_trend_keywords: dict,
    company_keywords: dict,
    days: int = 30,
    max_pages: int = 3
) -> pd.DataFrame:
    """
    섹터 키워드 + 회사 키워드 전체를 크롤링해서 news_df(DataFrame)로 반환.
    컬럼:
        date, company, keyword, keyword_group, keyword_type,
        title, summary, link, press, crawled_at
    """
    all_results = []

    # 1) 섹터 트렌드 키워드
    for group, keywords in sector_trend_keywords.items():  # group: "호재" / "악재"
        for kw in keywords:
            print(f"\n[섹터-{group}] 키워드 '{kw}' 크롤링 시작")
            news_list = crawl_naver_news(kw, days=days, max_pages=max_pages)
            for n in news_list:
                n["company"] = "SECTOR"          # 섹터 전체 이슈
                n["keyword_group"] = "sector"    # 섹터/회사 구분
                n["keyword_type"] = group        # "호재" / "악재"
            all_results.extend(news_list)

    # 2) 회사별 키워드 (회사명 + 게임명)
    for company, data in company_keywords.items():
        terms = data["company_terms"] + data["game_terms"]
        for kw in terms:
            print(f"\n[회사-{company}] 키워드 '{kw}' 크롤링 시작")
            news_list = crawl_naver_news(kw, days=days, max_pages=max_pages)
            for n in news_list:
                n["company"] = company
                n["keyword_group"] = "company"   # 섹터/회사 구분
                # company_terms인지 game_terms인지 구분
                n["keyword_type"] = (
                    "company_term" if kw in data["company_terms"] else "game_term"
                )
            all_results.extend(news_list)

    # 3) DataFrame으로 변환
    news_df = pd.DataFrame(all_results)

    if news_df.empty:
        print("❌ 크롤링 결과가 없습니다.")
        return news_df

    # 중복 제거 (링크 기준)
    if "link" in news_df.columns:
        before = len(news_df)
        news_df = news_df.drop_duplicates(subset=["link"], keep="first")
        print(f"\n중복 제거: {before} → {len(news_df)}")

    # content 컬럼 추가 (제목 + 요약 → NLP용)
    news_df["content"] = (
        news_df["title"].astype(str) + " " +
        news_df["summary"].fillna("").astype(str)
    ).str.strip()

    # date 정리
    news_df["date"] = pd.to_datetime(news_df["date"], errors="coerce").dt.date
    news_df = news_df.dropna(subset=["date"])

    print("\n=== 크롤링 최종 요약 ===")
    print("shape:", news_df.shape)
    print("\n회사별 기사 수:")
    print(news_df["company"].value_counts())

    return news_df


In [ ]:
# 전체 크롤링 실행 (Colab에서 이 셀 한번 돌리면 됨)
news_df = run_full_crawling(
    sector_trend_keywords=sector_trend_keywords,
    company_keywords=company_keywords,
    days=60,        # 최근 60일
    max_pages=3,    # 키워드당 최대 3페이지
)

news_df.head()



[섹터-호재] 키워드 '신작 출시' 크롤링 시작
[신작 출시] 페이지 1 요청 중...
  ⚠️ a.news_tit를 찾지 못했습니다. HTML 일부를 출력합니다.
<!DOCTYPE html>
<html lang="ko">
 <head>
  <meta charset="utf-8"/>
  <meta content="strict-origin-when-cross-origin" name="referrer"/>
  <meta content="telephone=no,address=no,email=no" name="format-detection"/>
  <meta content="신작 출시 : 네이버 뉴스검색" property="og:title">
   <meta content="https://ssl.pstatic.net/sstatic/search/common/og_v3.png" property="og:image"/>
   <meta content="'신작 출시'의 네이버 뉴스검색 결과입니다." property="og:description"/>
   <meta content="'신작 출시'의 네이버 뉴스검색 결과입니다." lang="ko" name="description"/>
   <title>
    신작 출시 : 네이버 뉴스검색
   </title>
   <link href="https://ssl.pstatic.net/sstatic/search/favicon/favicon_32x32_240820.ico" rel="shortcut icon"/>
   <link href="https://ssl.pstatic.net/sstatic/search/opensearch-description.https.xml" rel="search" title="Naver" type="application/o

[섹터-호재] 키워드 '출시 임박' 크롤링 시작
[출시 임박] 페이지 1 요청 중...
  ⚠️ a.news_tit를 찾지 못했습니다. HTML 일부를 출력합니다.
<!DOCTYPE htm

""


In [ ]:
https://finance.naver.com/item/news_news.naver?code=종목코드&page=페이지번호


SyntaxError: invalid syntax (ipython-input-1940882840.py, line 1)

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

def crawl_finance_item_news(code: str,
                            company_name: str,
                            max_pages: int = 5) -> pd.DataFrame:
    """
    네이버 금융 > 종목 뉴스 목록 크롤링 (테이블 구조 기준, 더 견고하게 수정 버전)
    code: 종목 코드 (e.g., '036570')
    company_name: 'NCSoft', 'Krafton', 'Mgame' 등
    """
    base_url = "https://finance.naver.com/item/news_news.naver"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    records = []

    for page in range(1, max_pages + 1):
        params = {
            "code": code,
            "page": page,
        }
        print(f"[{company_name}] 종목 뉴스 page {page} 요청 중...")

        resp = requests.get(base_url, headers=headers, params=params, timeout=10)
        resp.encoding = "euc-kr"  # 네이버 금융 인코딩

        if resp.status_code != 200:
            print(f"  ❌ status_code={resp.status_code}")
            break

        soup = BeautifulSoup(resp.text, "html.parser")

        # 보통 table.type5 이지만, 혹시 변동 있을 수 있으니 여유 있게 탐색
        table = soup.select_one("table.type5")
        if not table:
            table = soup.select_one("table.type2") or soup.find("table")

        if not table:
            print("  ⚠️ 뉴스 테이블(table) 자체를 찾지 못했습니다. HTML 일부를 출력합니다.")
            print(soup.prettify()[:800])
            break

        rows = table.find_all("tr")
        article_count = 0

        for tr in rows:
            tds = tr.find_all("td")
            if len(tds) < 3:
                continue  # 헤더/공백 행은 건너뜀

            a_tag = tds[0].find("a")
            if not a_tag or not a_tag.get("href"):
                continue  # 제목 링크 없으면 뉴스 아님

            title = a_tag.get_text(strip=True)
            # 상대 경로 → 절대 URL
            href = a_tag["href"]
            if href.startswith("/"):
                link = "https://finance.naver.com" + href
            else:
                link = href

            press = tds[1].get_text(strip=True)
            dt_str = tds[2].get_text(strip=True)  # '2025.01.10 09:30' 형태

            try:
                pub_dt = datetime.strptime(dt_str, "%Y.%m.%d %H:%M")
            except Exception:
                pub_dt = None

            records.append({
                "company": company_name,
                "code": code,
                "title": title,
                "summary": "",  # 나중에 필요하면 본문 크롤링 추가
                "link": link,
                "press": press,
                "datetime": pub_dt,
                "date": pub_dt.date() if pub_dt else None,
                "raw_dt": dt_str,
            })
            article_count += 1

        if article_count == 0:
            # 이 페이지에서 뉴스 row를 하나도 못 찾았으면, 여기서 멈춤
            print("  ⚠️ 이 페이지에서는 뉴스 row를 찾지 못했습니다. 테이블 HTML 일부를 출력합니다.")
            print(table.prettify()[:800])
            break
        else:
            print(f"  → 이 페이지에서 {article_count}건 수집")

    df = pd.DataFrame(records)
    print(f"  👉 {company_name} 뉴스 총 {len(df)}건 수집")
    return df


In [ ]:
code_map = {
    "NCSoft": "036570",
    "Krafton": "259960",
    "Mgame": "058630",
}

dfs = []
for company, code in code_map.items():
    df_c = crawl_finance_item_news(code, company_name=company, max_pages=5)
    if not df_c.empty:
        dfs.append(df_c)

if dfs:
    news_df = pd.concat(dfs, ignore_index=True)
    news_df["content"] = (
        news_df["title"].astype(str) + " " +
        news_df["summary"].fillna("").astype(str)
    ).str.strip()

    news_df["date"] = pd.to_datetime(news_df["date"], errors="coerce").dt.date
    news_df = news_df.dropna(subset=["date"])

    print("\n=== 종목 뉴스 기반 news_df 요약 ===")
    print("shape:", news_df.shape)
    print(news_df["company"].value_counts())
    display(news_df.head())
else:
    print("❌ 여전히 어떤 종목에서도 뉴스가 수집되지 않았습니다.")


[NCSoft] 종목 뉴스 page 1 요청 중...
  ⚠️ 이 페이지에서는 뉴스 row를 찾지 못했습니다. 테이블 HTML 일부를 출력합니다.
<table cellspacing="0" class="type5" summary="종목뉴스의 제목, 정보제공, 날짜">
 <caption>
  종목뉴스
 </caption>
 <colgroup>
  <col/>
  <col width="130px"/>
  <col width="110px"/>
 </colgroup>
 <thead>
  <tr>
   <th scope="col">
    제목
   </th>
   <th scope="col">
    정보제공
   </th>
   <th scope="col">
    날짜
   </th>
  </tr>
 </thead>
 <tbody>
  <tr>
   <td colspan="3">
    <div class="info_text_area">
     <p class="txt">
      <span class="ico">
      </span>
      최근 1년 내 검색된
      <em>
       ''
      </em>
      뉴스가 없습니다.
     </p>
    </div>
   </td>
  </tr>
  <!-- [D] tr class : 첫번째 first, 마지막 last, 연관 뉴스 relation_tit, 연관 뉴스 목록 relation_lst -->
 </tbody>
</table>

  👉 NCSoft 뉴스 총 0건 수집
[Krafton] 종목 뉴스 page 1 요청 중...
  ⚠️ 이 페이지에서는 뉴스 row를 찾지 못했습니다. 테이블 HTML 일부를 출력합니다.
<table cellspacing="0" class="type5" summary="종목뉴스의 제목, 정보제공, 날짜">
 <caption>
  종목뉴스
 </caption>
 <colgroup>
  <col/>
  <col width="130px"/>
  <col w

In [ ]:
if not news_df.empty:
    filename = f"naver_game_news_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    news_df.to_csv(filename, index=False, encoding="utf-8-sig")
    print("저장 완료:", filename)


In [ ]:
# 셀 1: 섹터/회사 키워드 정의 + 검색용 키워드 생성

# 1) 섹터 트렌드 키워드 (질문에서 준 것)
sector_trend_keywords = {
    "호재": [
        "신작 출시", "출시 임박", "흥행", "매출 1위", "사전예약",
        "중국 판호", "판호 발급", "글로벌 론칭", "다운로드 1위", "유저수 증가",
        "IP 확장", "콘솔 진출", "멀티 플랫폼", "라이브 서비스",
        "인공지능 NPC", "AI 기술 적용", "그래픽 업그레이드"
    ],
    "악재": [
        "흥행 부진", "매출 감소", "유저 이탈", "과금 논란", "확률형 아이템 논란",
        "규제 강화", "판호 불허", "심의 보류",
        "점검 논란", "서버 장애", "오류 발생",
        "리뷰 폭격", "평점 하락", "환불 사태"
    ]
}

# 2) 회사별 키워드 (질문에서 준 것)
company_keywords = {
    "NCSoft": {
        "company_terms": ["엔씨소프트", "NC소프트", "NCSoft"],
        "game_terms": [
            "리니지", "리니지M", "리니지W", "리니지2M",
            "아이온", "아이온2", "블레이드앤소울", "블레이드 & 소울",
            "프로젝트 LLL", "LLL"
        ],
    },
    "Krafton": {
        "company_terms": ["크래프톤", "KRAFTON"],
        "game_terms": [
            "배틀그라운드", "배그", "PUBG", "PUBG MOBILE",
            "다크앤다커", "Dark and Darker", "Dark and Darker Mobile",
            "inZOI", "인조이"
        ],
    },
    "Mgame": {
        "company_terms": ["엠게임", "Mgame"],
        "game_terms": [
            "열혈강호", "열혈강호 온라인", "열혈강호M",
            "나이트 온라인", "Knight Online"
        ],
    },
}

# 3) 크롤링에 쓸 검색 키워드 구성

# 3-1) 섹터 검색용 키워드: "게임주", "게임 섹터" + 섹터 트렌드 단어들
sector_search_keywords = (
    ["게임주", "게임 섹터", "모바일 게임", "온라인 게임"] +
    sector_trend_keywords["호재"] +
    sector_trend_keywords["악재"]
)

# 3-2) 회사별 검색용 키워드: 회사명 + 대표 게임명
company_search_keywords: Dict[str, List[str]] = {}
for company, cfg in company_keywords.items():
    kws = set(cfg["company_terms"] + cfg["game_terms"])
    company_search_keywords[company] = sorted(kws)

company_search_keywords

{'NCSoft': ['LLL',
  'NCSoft',
  'NC소프트',
  '리니지',
  '리니지2M',
  '리니지M',
  '리니지W',
  '블레이드 & 소울',
  '블레이드앤소울',
  '아이온',
  '아이온2',
  '엔씨소프트',
  '프로젝트 LLL'],
 'Krafton': ['Dark and Darker',
  'Dark and Darker Mobile',
  'KRAFTON',
  'PUBG',
  'PUBG MOBILE',
  'inZOI',
  '다크앤다커',
  '배그',
  '배틀그라운드',
  '인조이',
  '크래프톤'],
 'Mgame': ['Knight Online',
  'Mgame',
  '나이트 온라인',
  '엠게임',
  '열혈강호',
  '열혈강호 온라인',
  '열혈강호M']}

In [ ]:
# 셀 2: 네이버 뉴스 날짜 텍스트 → date 변환 함수

def parse_naver_news_date(date_str: str, fallback_today: bool = True) -> Optional[date]:
    """
    네이버 뉴스 검색 결과의 날짜 텍스트를 date로 변환.
    예)
      - '2025.01.10.'
      - '2025.01.10. 오전 9:30'
      - '3시간 전', '2일 전'
    """
    if not isinstance(date_str, str):
        date_str = str(date_str)
    date_str = date_str.strip()
    if not date_str:
        return date.today() if fallback_today else None

    # 1) 절대 날짜: '2025.01.10.' or '2025.01.10. 오전 9:30'
    try:
        base = date_str.split()[0]  # '2025.01.10.'
        base = base.rstrip(".")
        dt = datetime.strptime(base, "%Y.%m.%d")
        return dt.date()
    except Exception:
        pass

    # 2) 상대 표현: '3시간 전', '2일 전', '30분 전'
    now = datetime.now()
    try:
        if "분 전" in date_str:
            mins = int(date_str.replace("분 전", "").strip())
            return (now - timedelta(minutes=mins)).date()
        if "시간 전" in date_str:
            hrs = int(date_str.replace("시간 전", "").strip())
            return (now - timedelta(hours=hrs)).date()
        if "일 전" in date_str:
            days = int(date_str.replace("일 전", "").strip())
            return (now - timedelta(days=days)).date()
    except Exception:
        pass

    # 3) 파싱 실패 시
    return date.today() if fallback_today else None


In [ ]:
# 셀 3: 네이버 뉴스 크롤러 클래스 정의

class NaverNewsCrawler:
    def __init__(self):
        self.headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            )
        }
        self.base_url = "https://search.naver.com/search.naver"

    def search_news(self,
                    keyword: str,
                    days: int = 30,
                    max_pages: int = 3) -> List[Dict]:
        """
        네이버 '뉴스' 탭에서 keyword로 검색해서 기사 목록 크롤링.

        Args:
            keyword  : 검색 키워드
            days     : 최근 며칠까지 (기본 30일)
            max_pages: 최대 페이지 수 (페이지당 최대 10건)
        """
        news_list: List[Dict] = []

        end_date = datetime.now()
        start_date = end_date - timedelta(days=days)

        for page in range(1, max_pages + 1):
            start = (page - 1) * 10 + 1

            params = {
                "where": "news",
                "query": keyword,
                "start": start,
                "sort": 0,  # 0: 관련도순, 1: 최신순
                "pd": 3,    # 기간 직접 설정
                "ds": start_date.strftime("%Y.%m.%d"),
                "de": end_date.strftime("%Y.%m.%d"),
            }

            try:
                print(f"[{keyword}] 페이지 {page} 요청 중...")
                resp = requests.get(
                    self.base_url,
                    params=params,
                    headers=self.headers,
                    timeout=10
                )
                resp.raise_for_status()
                soup = BeautifulSoup(resp.text, "html.parser")

                # 다양한 셀렉터 시도
                articles = soup.select("div.news_area")
                if not articles:
                    articles = soup.select("ul.list_news > li")
                if not articles:
                    articles = soup.select("div.group_news > ul.list_news > li")

                if not articles:
                    print(f"  ⚠️ 페이지 {page}에서 기사 블록을 찾지 못했습니다.")
                    break

                print(f"  → {len(articles)}개 기사 발견")

                for art in articles:
                    try:
                        # 제목 & 링크
                        title_tag = (
                            art.select_one("a.news_tit")
                            or art.select_one("a.dsc_link")
                            or art.select_one("a")
                        )
                        if not title_tag:
                            continue

                        title = title_tag.get_text(strip=True)
                        link = title_tag.get("href", "")

                        # 요약
                        summary_tag = (
                            art.select_one("div.news_dsc")
                            or art.select_one("div.dsc_wrap")
                        )
                        summary = summary_tag.get_text(strip=True) if summary_tag else ""

                        # 언론사
                        press_tag = (
                            art.select_one("a.info.press")
                            or art.select_one("span.press")
                            or art.select_one("span[class*='press']")
                        )
                        press = press_tag.get_text(strip=True) if press_tag else ""

                        # 날짜 문자열
                        date_tag = (
                            art.select_one("span.info")
                            or art.select_one("span[class*='date']")
                        )
                        date_str = date_tag.get_text(strip=True) if date_tag else ""
                        pub_date = parse_naver_news_date(date_str)

                        news_list.append({
                            "keyword": keyword,
                            "title": title,
                            "summary": summary,
                            "link": link,
                            "press": press,
                            "date": pub_date,
                            "date_str": date_str,
                            "crawled_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        })

                    except Exception as e:
                        print(f"   기사 파싱 중 오류: {e}")
                        continue

                time.sleep(2)  # 페이지 간 딜레이

            except Exception as e:
                print(f"  X 페이지 {page} 요청/파싱 오류: {e}")
                break

        return news_list

    # 회사별 키워드 리스트를 받아서 크롤링하는 헬퍼
    def crawl_company(self,
                      company: str,
                      keywords: List[str],
                      days: int = 30,
                      max_pages: int = 3) -> pd.DataFrame:
        all_news: List[Dict] = []
        print(f"\n{'='*60}")
        print(f" 회사: {company}")
        print(f"{'='*60}")

        for kw in keywords:
            print(f"\n 키워드: '{kw}'")
            news = self.search_news(kw, days=days, max_pages=max_pages)
            for n in news:
                n["company"] = company
            all_news.extend(news)
            time.sleep(3)  # 키워드 간 딜레이

        df = pd.DataFrame(all_news)
        if not df.empty:
            # 링크 기준 중복 제거
            if "link" in df.columns:
                before = len(df)
                df = df.drop_duplicates(subset=["link"], keep="first")
                print(f"\n {company} 완료: {before} → {len(df)} (중복 제거)")
        else:
            print(f"\n {company}에서 수집된 기사가 없습니다.")

        return df

    # 섹터 트렌드 키워드 전체에 대해 크롤링
    def crawl_sector(self,
                     sector_keywords: List[str],
                     days: int = 30,
                     max_pages: int = 3) -> pd.DataFrame:
        all_news: List[Dict] = []
        print(f"\n{'='*60}")
        print(" 게임 섹터 트렌드 뉴스 크롤링 시작")
        print(f"{'='*60}")

        for kw in sector_keywords:
            print(f"\n [섹터] 키워드: '{kw}'")
            news = self.search_news(kw, days=days, max_pages=max_pages)
            for n in news:
                n["company"] = "SECTOR"  # 섹터 전체 이슈 표시
            all_news.extend(news)
            time.sleep(3)

        df = pd.DataFrame(all_news)
        if not df.empty:
            if "link" in df.columns:
                before = len(df)
                df = df.drop_duplicates(subset=["link"], keep="first")
                print(f"\n 섹터 뉴스 완료: {before} → {len(df)} (중복 제거)")
        else:
            print("\n 섹터 크롤링 결과가 없습니다.")

        return df


In [ ]:
# 셀 4: 크롤러 실행 → sector_news_df + company_news_df → news_df

crawler = NaverNewsCrawler()

DAYS = 60       # 최근 60일
MAX_PAGES = 3   # 키워드당 최대 3페이지

# 1) 섹터 트렌드 뉴스 크롤링
sector_news_df = crawler.crawl_sector(
    sector_keywords=sector_search_keywords,
    days=DAYS,
    max_pages=MAX_PAGES,
)

# 2) 회사별 뉴스 크롤링
company_dfs = []
for company, kws in company_search_keywords.items():
    df_c = crawler.crawl_company(
        company=company,
        keywords=kws,
        days=DAYS,
        max_pages=MAX_PAGES,
    )
    if not df_c.empty:
        company_dfs.append(df_c)

# 3) 섹터 + 회사 뉴스 통합
all_dfs = []
if sector_news_df is not None and not sector_news_df.empty:
    all_dfs.append(sector_news_df)
all_dfs += company_dfs

if all_dfs:
    news_df = pd.concat(all_dfs, ignore_index=True)

    # content 컬럼: 제목 + 요약
    news_df["content"] = (
        news_df.get("title", "").astype(str) + " " +
        news_df.get("summary", "").fillna("").astype(str)
    ).str.strip()

    # date 컬럼 정리 (이미 date로 넣었지만 혹시 모를 타입 변환)
    news_df["date"] = pd.to_datetime(news_df["date"], errors="coerce").dt.date
    news_df = news_df.dropna(subset=["date"])

    print("\n==================== 전체 news_df 요약 ====================")
    print("shape:", news_df.shape)
    print("\n회사별 기사 수:")
    print(news_df["company"].value_counts())

    display(
        news_df[["date", "company", "keyword", "title", "press"]]
        .head(10)
    )
else:
    print("X 수집된 뉴스가 없습니다. (크롤링 파라미터나 키워드를 확인해 주세요.)")



 게임 섹터 트렌드 뉴스 크롤링 시작

 [섹터] 키워드: '게임주'
[게임주] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '게임 섹터'
[게임 섹터] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '모바일 게임'
[모바일 게임] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '온라인 게임'
[온라인 게임] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '신작 출시'
[신작 출시] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '출시 임박'
[출시 임박] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '흥행'
[흥행] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '매출 1위'
[매출 1위] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '사전예약'
[사전예약] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '중국 판호'
[중국 판호] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '판호 발급'
[판호 발급] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '글로벌 론칭'
[글로벌 론칭] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '다운로드 1위'
[다운로드 1위] 페이지 1 요청 중...
  ⚠️ 페이지 1에서 기사 블록을 찾지 못했습니다.

 [섹터] 키워드: '유저수 증가'
[유저수 증가] 페이지 1 요청 중...


▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲ ▲  크롤링 완

In [ ]:
#
# 1) 설정: 종목, 피처 설계, 타깃 정의
#

stock_feature_config = {
    "universe": ["NCSoft", "Krafton", "Mgame"],
    "codes": {          # FinanceDataReader용 종목 코드
        "NCSoft": "036570",
        "Krafton": "259960",
        "Mgame": "058630",
    },
    "common": {
        "price_technical": [
            "close", "open", "high", "low", "volume",
            "ret_1d", "ret_3d", "ret_5d",
            "vol_5d",
            "ma_5", "ma_20", "ma_60",
            "ma_5_gap", "ma_20_gap", "ma_60_gap",
        ],
    },
    "by_company": {
        "NCSoft": {
            "text": {
                "pos_keywords": [
                    "어닝 서프라이즈", "흑자 전환", "실적 개선", "매출 1위", "흥행",
                    "사전예약", "글로벌 동시 출시", "판호 획득", "호평"
                ],
                "neg_keywords": [
                    "출시 연기", "매출 감소", "적자", "실적 부진",
                    "과금 논란", "유저 이탈", "보이콧", "논란", "규제"
                ],
            }
        },
        "Krafton": {
            "text": {
                "pos_keywords": [
                    "흥행", "동접자 증가", "매출 성장",
                    "글로벌 인기", "콘솔 성공", "AI First", "GPU 클러스터"
                ],
                "neg_keywords": [
                    "규제", "서비스 중단", "밴", "제재",
                    "동접자 감소", "매출 감소", "혹평", "출시 연기"
                ],
            }
        },
        "Mgame": {
            "text": {
                "pos_keywords": [
                    "역주행", "흥행", "역대 최대 매출",
                    "성장", "해외 성과", "장수 게임", "충성 유저"
                ],
                "neg_keywords": [
                    "계약 종료", "서비스 종료", "매출 감소",
                    "유저 감소", "리스크", "규제", "의존도"
                ],
            }
        },
    },
    "target": {
        "type": "classification",
        "horizon_days": 5,          # 5일 후
        "label_name": "ret_5d_up",  # 5일 후 수익률 > 0 ? 1 : 0
    }
}


In [ ]:
#
# 2) 주가 데이터 로딩 (FinanceDataReader)
#

!pip install finance-datareader --quiet

import FinanceDataReader as fdr
import pandas as pd
import numpy as np
from typing import Dict

def load_price_data(config: dict, start: str = "2022-01-01") -> Dict[str, pd.DataFrame]:
    """
    FinanceDataReader로 종목별 주가 데이터 불러오기.
    반환: { "NCSoft": DataFrame, "Krafton": DataFrame, ... }
    """
    codes = config["codes"]
    price_dict = {}

    for name, code in codes.items():
        print(f"{name} ({code}) 다운로드 중...")
        df = fdr.DataReader(code, start)
        if df is None or df.empty:
            print(f" {name} 데이터가 비어 있습니다. (코드/기간 확인 필요)")
            continue

        # 컬럼 이름 통일 (대부분 'Close' 그대로지만 안전하게)
        df = df.rename(columns=str.lower)
        price_dict[name] = df

    return price_dict


In [ ]:
#
# 3) 가격/ 기술 피쳐링
#
def add_price_technical_features_single(df: pd.DataFrame) -> pd.DataFrame:
    """
    단일 종목의 일별 가격 DataFrame에 기술적 피처 컬럼 추가.
    df: index = Date, columns에 'close', 'open', 'high', 'low', 'volume' 포함
    """
    df = df.copy()

    # 수익률
    df["ret_1d"] = df["close"].pct_change(1)
    df["ret_3d"] = df["close"].pct_change(3)
    df["ret_5d"] = df["close"].pct_change(5)

    # 변동성 (rolling std of returns)
    df["vol_5d"] = df["ret_1d"].rolling(window=5).std()

    # 이동평균
    df["ma_5"] = df["close"].rolling(window=5).mean()
    df["ma_20"] = df["close"].rolling(window=20).mean()
    df["ma_60"] = df["close"].rolling(window=60).mean()

    # MA 갭
    df["ma_5_gap"] = (df["close"] - df["ma_5"]) / df["ma_5"]
    df["ma_20_gap"] = (df["close"] - df["ma_20"]) / df["ma_20"]
    df["ma_60_gap"] = (df["close"] - df["ma_60"]) / df["ma_60"]

    return df


def build_price_feature_table(price_dict: Dict[str, pd.DataFrame],
                              config: dict) -> pd.DataFrame:
    """
    종목별 가격/기술 피처를 하나의 long-form 테이블로 합치기.
    반환: index = Date, columns = MultiIndex or wide-form (회사별 prefix)
    여기서는 long-form: [date, company, feature들...]
    """
    all_list = []

    for name, df in price_dict.items():
        df_feat = add_price_technical_features_single(df)

        df_feat = df_feat.reset_index().rename(columns={"index": "date"})
        df_feat["company"] = name

        all_list.append(df_feat)

    full_df = pd.concat(all_list, ignore_index=True)

    # 필요 없다면 config["common"]["price_technical"]만 선택해도 됨
    return full_df


In [ ]:
#
# 4) 텍스트 스코어링
#
"""
뉴스/토론방 DataFrame 가정

news_df columns 예시:
- date (datetime 또는 str)
- company (NCSoft / Krafton / Mgame)
- title (제목)
- content (본문, 옵션)
"""
def score_text_with_keywords(text: str,
                             pos_keywords: list,
                             neg_keywords: list,
                             pos_weight: float = 1.0,
                             neg_weight: float = -1.0) -> float:
    """
    한 문장/문서에 대해, 호재/악재 키워드 기반 점수 계산.
    (아주 단순한 count 기반 버전)
    """
    if not isinstance(text, str) or text.strip() == "":
        return 0.0

    t = text.lower()  # 대소문자 통일 (한글은 영향 적음)

    score = 0.0
    for w in pos_keywords:
        if w in t:
            score += pos_weight
    for w in neg_keywords:
        if w in t:
            score += neg_weight

    return score



In [ ]:
#
# 5) 일자, 회사별로 뉴스 점수 집계
#
def aggregate_daily_news_scores(news_df: pd.DataFrame,
                                config: dict) -> pd.DataFrame:
    """
    뉴스 DataFrame에서 회사별, 날짜별 호재/악재 점수 집계.
    news_df: [date, company, title, content]
    반환: [date, company, news_count, news_score_sum, news_score_avg]
    """

    # 0) news_df가 비었으면 바로 빈 결과 반환
    if news_df is None or news_df.empty:
        return pd.DataFrame(columns=["date", "company",
                                     "news_count", "news_score_sum", "news_score_avg"])

    df = news_df.copy()

    # date 컬럼 정리
    df["date"] = pd.to_datetime(df["date"]).dt.date

    # title, content 없으면 기본 컬럼 추가
    if "title" not in df.columns:
        df["title"] = ""
    if "content" not in df.columns:
        df["content"] = ""

    rows = []

    for company in config["universe"]:
        c_conf = config["by_company"][company]["text"]
        pos_kw = c_conf["pos_keywords"]
        neg_kw = c_conf["neg_keywords"]

        sub = df[df["company"] == company].copy()
        if sub.empty:
            # 이 회사에 대한 뉴스가 없으면 스킵
            continue

        #  제목 + 본문 단순 문자열 더하기 (agg 대신)
        sub["text_all"] = sub["title"].fillna("") + " " + sub["content"].fillna("")

        # 점수 계산
        sub["score"] = sub["text_all"].apply(
            lambda x: score_text_with_keywords(x, pos_kw, neg_kw)
        )

        # 일자별 집계
        daily = sub.groupby("date").agg(
            news_count=("score", "count"),
            news_score_sum=("score", "sum"),
            news_score_avg=("score", "mean"),
        ).reset_index()

        daily["company"] = company
        rows.append(daily)

    # 회사별로 다 비어 있었으면 빈 DF 반환
    if not rows:
        return pd.DataFrame(columns=["date", "company",
                                     "news_count", "news_score_sum", "news_score_avg"])

    daily_scores = pd.concat(rows, ignore_index=True)
    return daily_scores


In [ ]:
#
# 6) 가격 + 뉴스 피처 merge & 타깃 생성
#
#
def add_target_label(price_feature_df: pd.DataFrame,
                     horizon_days: int = 5) -> pd.DataFrame:
    """
    long-form [date, company, close, ...] 에서
    회사별로 horizon_days 후 수익률 기반 타깃 레이블 생성.
    - future_price: N일 후 종가
    - future_ret:   N일 후 수익률
    - ret_{N}_up:   수익률 > 0 이면 1, 아니면 0
    """
    df = price_feature_df.copy()
    df["date"] = pd.to_datetime(df["date"])

    out_list = []
    for name, sub in df.groupby("company"):
        sub = sub.sort_values("date")
        sub["future_price"] = sub["close"].shift(-horizon_days)
        sub["future_ret"] = (sub["future_price"] - sub["close"]) / sub["close"]
        sub[f"ret_{horizon_days}_up"] = (sub["future_ret"] > 0).astype(int)
        out_list.append(sub)

    df_out = pd.concat(out_list, ignore_index=True)
    return df_out


def build_training_table(price_feature_df: pd.DataFrame,
                         news_feature_df: pd.DataFrame,
                         config: dict) -> pd.DataFrame:
    """
    가격 피처와 뉴스 피처를 날짜+회사 기준으로 merge하고,
    타깃까지 붙여서 최종 학습용 테이블을 만든다.
    """

    df_price = price_feature_df.copy()
    df_news = news_feature_df.copy()

    # 날짜 포맷 통일 (date 컬럼을 date 타입으로)
    df_price["date"] = pd.to_datetime(df_price["date"]).dt.date
    if not df_news.empty:
        df_news["date"] = pd.to_datetime(df_news["date"]).dt.date

    # 가격 기준으로 뉴스 붙이기 (left join)
    merged = pd.merge(
        df_price,
        df_news,
        on=["date", "company"],
        how="left"
    )

    # 뉴스 없는 날은 0으로 채우기
    for col in ["news_count", "news_score_sum", "news_score_avg"]:
        if col in merged.columns:
            merged[col] = merged[col].astype("float64").fillna(0.0)

    # 타깃 생성
    horizon = config["target"]["horizon_days"]
    merged = add_target_label(merged, horizon_days=horizon)

    # 맨 끝 몇 일은 future_price가 없어서 NaN → 제거
    merged = merged.dropna(subset=["future_ret"])

    return merged



In [ ]:
#
# 6) 가격 + 뉴스 피처 merge & 타깃 생성
#
#
# 2. 가격 + 뉴스 피처 합치기 (최종 학습용 테이블)
#
def build_price_feature_table(price_dict: dict,
                              config: dict) -> pd.DataFrame:
    """
    종목별 가격/기술 피처를 하나의 long-form 테이블로 합치기.
    반환: [date, company, close, ret_1d, ...] 형태
    """
    all_list = []

    for name, df in price_dict.items():
        df_feat = add_price_technical_features_single(df)

        #  index 이름이 Date든 뭐든 간에, 첫 번째 컬럼을 date로 통일
        df_feat = df_feat.reset_index()
        first_col_name = df_feat.columns[0]      # 보통 'Date' 또는 'index'
        df_feat = df_feat.rename(columns={first_col_name: "date"})

        df_feat["company"] = name
        all_list.append(df_feat)

    full_df = pd.concat(all_list, ignore_index=True)
    return full_df


In [ ]:
# 시각화
import plotly.express as px
import pandas as pd

# 날짜 타입 정리 (혹시 몰라서)
price_feat_df["date"] = pd.to_datetime(price_feat_df["date"])

fig_price = px.line(
    price_feat_df,
    x="date",
    y="close",
    color="company",
    title="엔씨소프트 / 크래프톤 / 엠게임 - 일별 종가",
    labels={"date": "날짜", "close": "종가", "company": "종목"}
)
fig_price.update_layout(legend_title_text="종목")
fig_price.show()


NameError: name 'price_feat_df' is not defined

In [ ]:
# 기준일: 각 종목의 첫 날
price_norm = price_feat_df.copy()
price_norm = price_norm.sort_values(["company", "date"])

price_norm["base_close"] = price_norm.groupby("company")["close"].transform("first")
price_norm["close_indexed"] = price_norm["close"] / price_norm["base_close"] * 100

fig_norm = px.line(
    price_norm,
    x="date",
    y="close_indexed",
    color="company",
    title="엔씨소프트 / 크래프톤 / 엠게임 - 수익률 흐름 (첫 날 = 100)",
    labels={"date": "날짜", "close_indexed": "지수화된 가격", "company": "종목"}
)
fig_norm.update_layout(legend_title_text="종목")
fig_norm.show()


In [ ]:
#

# 1) 주가 데이터 로딩
price_dict = load_price_data(stock_feature_config, start="2020-01-01")

# 2) 가격/기술 피처 테이블 생성
price_feat_df = build_price_feature_table(price_dict, stock_feature_config)

# 3) 뉴스 DataFrame 가정 (나중에 크롤링/전처리해서 맞춰 넣기)
# news_df = pd.DataFrame([...])  # columns: [date, company, title, content]

# 일단 예시용 빈 DF
news_df = pd.DataFrame(columns=["date", "company", "title", "content"])

# 4) 뉴스 키워드 점수 집계
news_feat_df = aggregate_daily_news_scores(news_df, stock_feature_config)

# 5) 최종 학습용 테이블
train_df = build_training_table(price_feat_df, news_feat_df, stock_feature_config)

train_df.head()


NCSoft (036570) 다운로드 중...
Krafton (259960) 다운로드 중...
Mgame (058630) 다운로드 중...


KeyError: 'date'

In [ ]:
# train_df 기준 – 미래 수익률 분포 보기
# 5일 후 수익률 분포 (연속값 느낌 파악)

train_df["future_ret_pct"] = train_df["future_ret"] * 100  # %로 보기

fig_ret = px.histogram(
    train_df,
    x="future_ret_pct",
    nbins=50,
    color="company",
    marginal="box",          # 위에 박스플롯 같이 표시
    title="5일 후 수익률 분포 (회사별)",
    labels={"future_ret_pct": "5일 후 수익률 (%)", "company": "종목"}
)
fig_ret.update_layout(legend_title_text="종목")
fig_ret.show()


In [ ]:
# 라벨(상승/하락) 비율 확인 (ret_5_up)

fig_label = px.histogram(
    train_df,
    x="ret_5_up",
    color="company",
    barmode="group",
    title="5일 후 상승(1) / 하락(0) 라벨 분포",
    labels={"ret_5_up": "라벨 (0=하락, 1=상승)", "company": "종목"}
)
fig_label.update_xaxes(tickmode="array", tickvals=[0, 1], ticktext=["하락", "상승"])
fig_label.update_layout(legend_title_text="종목")
fig_label.show()


In [ ]:
# ma_20_gap(20일선 대비 괴리율) vs future_ret 스캐터
# 20일선 위로 많이 떠 있을 때는 이후 수익률이 안 좋다” 같은 패턴이 있는지

fig_scatter = px.scatter(
    train_df,
    x="ma_20_gap",
    y="future_ret",
    color="company",
    title="20일선 갭 vs 5일 후 수익률",
    labels={"ma_20_gap": "MA20 갭 (현재가 대비)", "future_ret": "5일 후 수익률", "company": "종목"},
    opacity=0.5
)
fig_scatter.update_layout(legend_title_text="종목")
fig_scatter.show()


In [ ]:
# 대표 기술 지표 vs  5일 후 수익률

import plotly.express as px

# 보고 싶은 기술적 지표들 목록
tech_cols = [
    "ma_5_gap",    # 5일선 대비 괴리율
    "ma_20_gap",   # 20일선 대비 괴리율
    "ma_60_gap",   # 60일선 대비 괴리율
    "ret_1d",      # 전일 수익률
    "vol_5d",      # 5일 변동성
]

# NaN 제거 (특히 초반 구간)
plot_df = train_df.copy()
plot_df = plot_df.dropna(subset=tech_cols + ["future_ret"])

for col in tech_cols:
    print(f"▶ {col} vs future_ret 스캐터")

    fig = px.scatter(
        plot_df,
        x=col,
        y="future_ret",
        color="company",          # 회사별 색깔 구분
        opacity=0.4,
        title=f"{col} vs 5일 후 수익률 (future_ret)",
        labels={
            col: col,
            "future_ret": "5일 후 수익률",
            "company": "종목"
        }
    )
    fig.update_layout(legend_title_text="종목")
    fig.show()


NameError: name 'train_df' is not defined

In [ ]:
fig = px.scatter(
    plot_df,
    x="ma_20_gap",
    y="future_ret",
    color="company",
    opacity=0.4,
    title="20일선 갭 vs 5일 후 수익률",
    labels={
        "ma_20_gap": "MA20 갭 (현재가 대비)",
        "future_ret": "5일 후 수익률",
        "company": "종목"
    }
)
fig.update_layout(legend_title_text="종목")
fig.show()


In [ ]:
fig = px.scatter(
    plot_df,
    x="ma_20_gap",
    y="future_ret",
    color="company",
    facet_col="company",   # 회사별로 칸 나눠서 보기
    facet_col_wrap=3,
    opacity=0.4,
    title="20일선 갭 vs 5일 후 수익률 (회사별 패널)",
    labels={
        "ma_20_gap": "MA20 갭 (현재가 대비)",
        "future_ret": "5일 후 수익률",
        "company": "종목"
    }
)
fig.update_layout(legend_title_text="종목")
fig.show()


In [ ]:
cols = ["ret_1d", "vol_5d"]
print(train_df[cols + ["future_ret"]].corr()["future_ret"])


ret_1d       -0.050923
vol_5d        0.048249
future_ret    1.000000
Name: future_ret, dtype: float64


In [ ]:
# ret_1d를 구간(quantile)으로 나눠서, 각 구간별 future_ret 평균 보기
train_df["ret_1d_bin"] = pd.qcut(train_df["ret_1d"], q=10, duplicates="drop")

ret_bin_mean = train_df.groupby("ret_1d_bin")["future_ret"].mean()
display(ret_bin_mean)


/tmp/ipython-input-1003104749.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,future_ret
ret_1d_bin,
"(-0.195, -0.0308]",0.010573
"(-0.0308, -0.0186]",0.005127
"(-0.0186, -0.0112]",-0.003974
"(-0.0112, -0.00564]",-0.001106
"(-0.00564, 0.0]",-0.001485
"(0.0, 0.00443]",0.001169
"(0.00443, 0.0102]",0.002818
"(0.0102, 0.0189]",0.002899
"(0.0189, 0.0325]",-0.004857


In [ ]:
train_df["ret_1d_extreme_down"] = (train_df["ret_1d"] <= train_df["ret_1d"].quantile(0.1)).astype(int)
train_df["ret_1d_extreme_up"]   = (train_df["ret_1d"] >= train_df["ret_1d"].quantile(0.9)).astype(int)


In [ ]:
import plotly.express as px

# 1) 구간별 future_ret 평균 계산 (observed=False로 경고 제거)
ret_bin_mean = (
    train_df
    .groupby("ret_1d_bin", observed=False)["future_ret"]
    .mean()
    .reset_index()
)

# 2) Interval 타입을 문자열로 변환 (Plotly용)
ret_bin_mean["ret_1d_bin_str"] = ret_bin_mean["ret_1d_bin"].astype(str)

# 3) 막대그래프 그리기
fig = px.bar(
    ret_bin_mean,
    x="ret_1d_bin_str",
    y="future_ret",
    title="전일 수익률 구간별 5일 후 평균 수익률",
    labels={
        "ret_1d_bin_str": "전일 수익률 구간",
        "future_ret": "5일 후 평균 수익률",
    },
)
fig.update_layout(xaxis_tickangle=-45)  # 라벨이 겹치면 살짝 기울여 주기
fig.show()


크롤링 해보자


In [ ]:
# 검색어 기반 크롤링

#  필요 라이브러리

!pip install requests beautifulsoup4 lxml --quiet

import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta, date
from typing import Optional
import time
import pandas as pd


In [ ]:
# 검색어 기반 크롤링

# 2 검색어

stock_feature_config["search_keywords"] = {
    "NCSoft": [
        # 회사명
        "엔씨소프트", "NC소프트", "NCSoft",
        # 대표 IP / 게임명
        "리니지", "리니지M", "리니지W", "리니지2M",
        "아이온", "아이온2", "AION 2",
        "블레이드앤소울", "블레이드 & 소울",
        "LLL", "프로젝트 LLL",
        "Journey of Monarch", "Horizon", "TACTAN"
    ],
    "Krafton": [
        # 회사명
        "크래프톤", "KRAFTON",
        # 대표 IP / 게임명
        "배틀그라운드", "배그", "PUBG", "PUBG MOBILE",
        "다크앤다커", "Dark and Darker", "Dark and Darker Mobile",
        "인조이", "inZOI"
    ],
    "Mgame": [
        # 회사명
        "엠게임", "Mgame",
        # 대표 IP / 게임명
        "열혈강호", "열혈강호 온라인", "열혈강호M",
        "나이트 온라인", "Knight Online",
        "진명강호", "Zhenmin Jianghu"
    ]
}


In [ ]:
def parse_naver_news_date(date_str: str, fallback_today: bool = True) -> Optional[date]:
    """
    네이버 뉴스 검색 결과의 날짜 텍스트를 date로 변환.
    - '2025.01.10.' 또는 '2025.01.10. 오전 9:30'
    - '3시간 전', '2일 전' 같은 상대 표현도 일부 처리
    - 파싱 실패 시 fallback_today=True이면 오늘 날짜 반환
    """
    date_str = str(date_str).strip()
    if not date_str:
        return date.today() if fallback_today else None

    # 1) 절대 날짜 (2025.01.10. 또는 2025.01.10. 오전 9:30)
    try:
        base = date_str.split()[0]   # '2025.01.10.' 까지만
        base = base.rstrip(".")      # 마지막 점 제거
        dt = datetime.strptime(base, "%Y.%m.%d")
        return dt.date()
    except Exception:
        pass

    # 2) 상대 날짜: '3시간 전', '2일 전', '30분 전' 등
    now = datetime.now()
    try:
        if "분 전" in date_str:
            mins = int(date_str.replace("분 전", "").strip())
            return (now - timedelta(minutes=mins)).date()
        if "시간 전" in date_str:
            hrs = int(date_str.replace("시간 전", "").strip())
            return (now - timedelta(hours=hrs)).date()
        if "일 전" in date_str:
            days = int(date_str.replace("일 전", "").strip())
            return (now - timedelta(days=days)).date()
    except Exception:
        pass

    # 3) 그 외 포맷은 fallback
    return date.today() if fallback_today else None


In [ ]:
# 검색어 기반 크롤링

# 네이버 “뉴스 검색” 기반 크롤러 (검색어 확장 버전)

NAVER_SEARCH_NEWS_URL = "https://search.naver.com/search.naver"

def crawl_naver_news_by_keyword(keyword: str,
                                company_label: str,
                                max_pages: int = 3,
                                start_date: str = "2022-01-01"):
    """
    네이버 '뉴스' 탭에서 특정 keyword로 검색한 기사 목록 크롤링.
    - keyword: 검색어 (예: '엔씨소프트', '배틀그라운드')
    - company_label: 이 뉴스들을 귀속시킬 회사 ('NCSoft', 'Krafton', 'Mgame')
    """
    records = []
    start_dt = datetime.strptime(start_date, "%Y-%m-%d").date()

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/118.0 Safari/537.36"
        )
    }

    for page in range(max_pages):
        start = page * 10 + 1  # 1, 11, 21, ...
        params = {
            "where": "news",
            "sm": "tab_pge",
            "query": keyword,
            "start": start,
        }

        resp = requests.get(NAVER_SEARCH_NEWS_URL, params=params, headers=headers)
        print(f"[{company_label} | {keyword}] page {page+1} status:", resp.status_code)
        if resp.status_code != 200:
            break

        soup = BeautifulSoup(resp.text, "lxml")

        # 1) 가장 일반적인 구조: div.news_area
        news_list = soup.select("div.news_area")

        # 2) 혹시 news_area를 못 찾으면, 느슨하게 a.news_tit만이라도 시도
        if not news_list:
            print("  -> div.news_area 0개, a.news_tit 기반으로 재시도")
            news_title_links = soup.select("a.news_tit")
            if not news_title_links:
                print("  -> a.news_tit도 0개, 이 페이지는 뉴스가 아닌 듯")
                break

            for a_tag in news_title_links:
                title = a_tag.get("title", "").strip()
                url = a_tag.get("href", "").strip()
                if not title:
                    continue

                # 날짜는 못 찾았으니 일단 오늘 날짜로 대체
                dt = date.today()
                if dt < start_dt:
                    continue

                records.append({
                    "date": dt,
                    "company": company_label,
                    "title": title,
                    "content": "",
                    "url": url,
                    "keyword": keyword,
                })

            # 이 경우는 더 깊이 파싱할 게 없으니 다음 페이지로 넘어감
            time.sleep(0.7)
            continue

        # 3) 정상적인 news_area 구조인 경우
        for area in news_list:
            a_tag = area.select_one("a.news_tit")
            if not a_tag:
                continue
            title = a_tag.get("title", "").strip()
            url = a_tag.get("href", "").strip()
            if not title:
                continue

            # 날짜 텍스트 찾기
            info_group = area.select_one("div.info_group")
            date_text = None
            if info_group:
                for span in info_group.select("span.info"):
                    t = span.get_text(strip=True)
                    # '언론사'가 아니라 '2025.01.10.' 또는 '3시간 전' 같은 것만
                    if "." in t or "전" in t:
                        date_text = t
                        break

            # 날짜 파싱 (실패하면 오늘 날짜)
            dt = parse_naver_news_date(date_text if date_text else "")

            # 너무 과거 데이터면 스킵만 하고, 전체 탐색은 계속
            if dt < start_dt:
                continue

            records.append({
                "date": dt,
                "company": company_label,
                "title": title,
                "content": "",
                "url": url,
                "keyword": keyword,
            })

        time.sleep(0.7)  # 요청 간격 넉넉히

    return records


In [ ]:
# 검색어 기반 크롤링

# 회사별(엔씨/크래프톤/엠게임) 키워드 확장 크롤링 → news_df 만들기

In [ ]:
# 필요한 라이브러리 설치
!pip install requests beautifulsoup4 pandas lxml selenium

import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import time
import urllib.parse
from typing import List, Dict
import re

# 검색 키워드 설정
stock_feature_config = {}
stock_feature_config["search_keywords"] = {
    "NCSoft": [
        "엔씨소프트", "NC소프트", "NCSoft",
        "리니지", "리니지M", "리니지W", "리니지2M",
        "아이온", "아이온2", "블레이드앤소울", "프로젝트 LLL",
    ],
    "Krafton": [
        "크래프톤", "KRAFTON",
        "배틀그라운드", "배그", "PUBG", "PUBG MOBILE",
        "다크앤다커", "Dark and Darker", "inZOI",
    ],
    "Mgame": [
        "엠게임", "Mgame",
        "열혈강호", "열혈강호M",
        "나이트 온라인", "Knight Online",
    ]
}

class NaverNewsCrawler:
    def __init__(self):
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        }
        self.base_url = "https://search.naver.com/search.naver"

    def search_news(self, keyword: str, days: int = 30, max_pages: int = 5) -> List[Dict]:
        """
        네이버 뉴스 검색 및 크롤링

        Args:
            keyword: 검색 키워드
            days: 최근 며칠 기사 (기본 30일)
            max_pages: 최대 페이지 수

        Returns:
            뉴스 정보 리스트
        """
        news_list = []

        # 날짜 계산
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days)

        for page in range(1, max_pages + 1):
            start = (page - 1) * 10 + 1

            params = {
                'where': 'news',
                'query': keyword,
                'start': start,
                'sort': 0,  # 0: 관련도순, 1: 최신순
                'pd': 3,  # 기간 직접 설정
                'ds': start_date.strftime('%Y.%m.%d'),
                'de': end_date.strftime('%Y.%m.%d')
            }

            try:
                print(f"'{keyword}' 페이지 {page} 요청 중...")
                response = requests.get(self.base_url, params=params, headers=self.headers, timeout=10)
                response.raise_for_status()

                soup = BeautifulSoup(response.text, 'html.parser')

                # 다양한 셀렉터 시도 (2024년 이후 변경된 구조 대응)
                articles = []

                # 시도 1: news_area 클래스
                articles = soup.select('div.news_area')

                # 시도 2: bx 클래스 (구버전)
                if not articles:
                    articles = soup.select('ul.list_news > li')

                # 시도 3: api_subject_bx (또 다른 구조)
                if not articles:
                    articles = soup.select('div.group_news > ul.list_news > li')

                # 시도 4: 가장 일반적인 구조
                if not articles:
                    articles = soup.select('div[class*="news"]')

                if not articles:
                    print(f"   페이지 {page}에서 기사를 찾을 수 없습니다. 셀렉터 확인 필요")

                    # 디버깅: HTML 구조 일부 출력
                    print("\n=== HTML 구조 샘플 (디버깅용) ===")
                    main_content = soup.find('div', {'id': 'main_pack'})
                    if main_content:
                        print(main_content.prettify()[:1000])
                    print("="*50)
                    break

                print(f"  ✓ {len(articles)}개 기사 발견")

                for idx, article in enumerate(articles):
                    try:
                        # 제목과 링크 추출 (여러 패턴 시도)
                        title_tag = (
                            article.select_one('a.news_tit') or
                            article.select_one('a.dsc_link') or
                            article.select_one('a[class*="tit"]') or
                            article.select_one('a')
                        )

                        if not title_tag:
                            continue

                        title = title_tag.get_text(strip=True)
                        link = title_tag.get('href', '')

                        if not title or not link:
                            continue

                        # 요약 추출
                        summary_tag = (
                            article.select_one('div.news_dsc') or
                            article.select_one('div.dsc_wrap') or
                            article.select_one('div[class*="dsc"]')
                        )
                        summary = summary_tag.get_text(strip=True) if summary_tag else ""

                        # 언론사 추출
                        press_tag = (
                            article.select_one('a.info.press') or
                            article.select_one('a.press') or
                            article.select_one('span[class*="press"]')
                        )
                        press = press_tag.get_text(strip=True) if press_tag else ""

                        # 날짜 추출
                        date_tag = (
                            article.select_one('span.info') or
                            article.select_one('span[class*="date"]')
                        )
                        date = date_tag.get_text(strip=True) if date_tag else ""

                        news_list.append({
                            'keyword': keyword,
                            'title': title,
                            'link': link,
                            'summary': summary,
                            'press': press,
                            'date': date,
                            'crawled_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                        })

                    except Exception as e:
                        print(f"   기사 {idx+1} 파싱 오류: {e}")
                        continue

                print(f"  → 페이지 {page} 완료: {len([n for n in news_list if n['keyword'] == keyword])}개 수집")

                # 빈 페이지면 중단
                if len(articles) == 0:
                    break

                time.sleep(2)  # 요청 간 딜레이 (중요!)

            except requests.exceptions.RequestException as e:
                print(f"   페이지 {page} 요청 오류: {e}")
                break
            except Exception as e:
                print(f"   페이지 {page} 처리 오류: {e}")
                break

        return news_list

    def crawl_by_company(self, company: str, keywords: List[str],
                        days: int = 30, max_pages: int = 3) -> pd.DataFrame:
        """
        회사별로 여러 키워드 검색

        Args:
            company: 회사명
            keywords: 검색 키워드 리스트
            days: 최근 며칠 기사
            max_pages: 키워드당 최대 페이지 수

        Returns:
            DataFrame
        """
        all_news = []

        print(f"\n{'='*60}")
        print(f" 회사: {company}")
        print(f"{'='*60}")

        for keyword in keywords:
            print(f"\n 키워드: '{keyword}'")
            news = self.search_news(keyword, days, max_pages)
            all_news.extend(news)
            time.sleep(3)  # 키워드 간 딜레이 (중요!)

        df = pd.DataFrame(all_news)

        if not df.empty:
            # 중복 제거 (동일한 링크)
            before_count = len(df)
            df = df.drop_duplicates(subset=['link'], keep='first')
            after_count = len(df)

            df['company'] = company
            print(f"\n {company} 완료")
            print(f"   총 수집: {before_count}개")
            print(f"   중복 제거 후: {after_count}개")
        else:
            print(f"\n⚠️ {company}에서 수집된 기사가 없습니다.")

        return df

# 크롤러 초기화
crawler = NaverNewsCrawler()

# 설정
DAYS = 30  # 최근 30일
MAX_PAGES = 3  # 키워드당 최대 페이지

print(f"\n{'='*60}")
print(f"네이버 뉴스 크롤링 시작")
print(f"기간: 최근 {DAYS}일")
print(f"페이지: 키워드당 최대 {MAX_PAGES}페이지")
print(f"{'='*60}")

# 전체 데이터를 저장할 리스트
all_dataframes = []

# 각 회사별로 크롤링
for company, keywords in stock_feature_config["search_keywords"].items():
    df = crawler.crawl_by_company(
        company=company,
        keywords=keywords,
        days=DAYS,
        max_pages=MAX_PAGES
    )

    if not df.empty:
        all_dataframes.append(df)

    time.sleep(5)  # 회사 간 딜레이

# 전체 데이터 통합
if all_dataframes:
    final_df = pd.concat(all_dataframes, ignore_index=True)

    # 결과 출력
    print(f"\n{'='*60}")
    print(f"전체 수집 결과")
    print(f"{'='*60}")
    print(f"총 기사 수: {len(final_df)}")
    print(f"\n회사별 기사 수:")
    company_counts = final_df['company'].value_counts()
    for company, count in company_counts.items():
        print(f"  • {company}: {count}개")

    # CSV 저장
    filename = f'naver_news_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    final_df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"\n 파일 저장 완료: {filename}")

    # 샘플 데이터 출력
    print(f"\n{'='*60}")
    print(" 샘플 데이터 (최근 5개)")
    print(f"{'='*60}")
    display_df = final_df[['company', 'keyword', 'title', 'press', 'date']].head()
    print(display_df.to_string(index=False))

    # Colab에서 파일 다운로드
    try:
        from google.colab import files
        files.download(filename)
        print(f"\n 파일 다운로드 시작!")
    except:
        print(f"\n Colab이 아닌 환경에서는 '{filename}' 파일을 직접 확인하세요.")
else:
    print("\n 수집된 데이터가 없습니다.")
    print("\n 문제 해결 방법:")
    print("  1. 인터넷 연결을 확인하세요")
    print("  2. 키워드를 더 일반적인 단어로 변경해보세요")
    print("  3. 기간(DAYS)을 늘려보세요")
    print("  4. 네이버 검색에서 실제로 검색되는지 확인하세요")


네이버 뉴스 크롤링 시작
기간: 최근 30일
페이지: 키워드당 최대 3페이지

🏢 회사: NCSoft

🔍 키워드: '엔씨소프트'
'엔씨소프트' 페이지 1 요청 중...
  ✓ 4개 기사 발견
  → 페이지 1 완료: 2개 수집
'엔씨소프트' 페이지 2 요청 중...
  ✓ 4개 기사 발견
  → 페이지 2 완료: 4개 수집
'엔씨소프트' 페이지 3 요청 중...
  ✓ 4개 기사 발견
  → 페이지 3 완료: 6개 수집

🔍 키워드: 'NC소프트'
'NC소프트' 페이지 1 요청 중...
  ✓ 4개 기사 발견
  → 페이지 1 완료: 2개 수집
'NC소프트' 페이지 2 요청 중...
  ✓ 4개 기사 발견
  → 페이지 2 완료: 4개 수집
'NC소프트' 페이지 3 요청 중...
  ✓ 4개 기사 발견
  → 페이지 3 완료: 6개 수집

🔍 키워드: 'NCSoft'
'NCSoft' 페이지 1 요청 중...
  ✓ 4개 기사 발견
  → 페이지 1 완료: 2개 수집
'NCSoft' 페이지 2 요청 중...
  ✓ 4개 기사 발견
  → 페이지 2 완료: 4개 수집
'NCSoft' 페이지 3 요청 중...
  ✓ 4개 기사 발견
  → 페이지 3 완료: 6개 수집

🔍 키워드: '리니지'
'리니지' 페이지 1 요청 중...
  ✓ 4개 기사 발견
  → 페이지 1 완료: 2개 수집
'리니지' 페이지 2 요청 중...
  ✓ 4개 기사 발견
  → 페이지 2 완료: 4개 수집
'리니지' 페이지 3 요청 중...
  ✓ 4개 기사 발견
  → 페이지 3 완료: 6개 수집

🔍 키워드: '리니지M'
'리니지M' 페이지 1 요청 중...
  ✓ 4개 기사 발견
  → 페이지 1 완료: 2개 수집
'리니지M' 페이지 2 요청 중...
  ✓ 4개 기사 발견
  → 페이지 2 완료: 4개 수집
'리니지M' 페이지 3 요청 중...
  ✓ 4개 기사 발견
  → 페이지 3 완료: 6개 수집

🔍 키워드: '리니지W'
'리니지W' 페이지 1 요청 중...
  ✓ 4개 기사

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ 파일 다운로드 시작!


In [ ]:
# 필요한 라이브러리 설치
!pip install pandas numpy konlpy matplotlib seaborn scikit-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

print("=" * 80)
print("뉴스 데이터 피처 엔지니어링")
print("=" * 80)

# ============================================
# 1. 데이터 로드 (news_df가 이미 있다고 가정)
# ============================================
print("\n📊 1. 데이터 확인")
print("-" * 80)

if 'news_df' not in globals():
    print("⚠️ news_df가 없습니다. CSV 파일을 업로드하세요.")
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    news_df = pd.read_csv(filename, encoding='utf-8-sig')

print(f"전체 뉴스 개수: {len(news_df):,}개")
print(f"컬럼: {list(news_df.columns)}")

# ============================================
# 2. 날짜 전처리
# ============================================
print("\n📅 2. 날짜 데이터 전처리")
print("-" * 80)

def parse_date(date_str):
    """네이버 뉴스 날짜 파싱"""
    if pd.isna(date_str):
        return None

    try:
        # "1시간 전", "3일 전" 등 처리
        if '시간 전' in str(date_str):
            hours = int(re.findall(r'\d+', str(date_str))[0])
            return datetime.now() - timedelta(hours=hours)
        elif '일 전' in str(date_str):
            days = int(re.findall(r'\d+', str(date_str))[0])
            return datetime.now() - timedelta(days=days)
        elif '분 전' in str(date_str):
            return datetime.now()
        else:
            # "2024.11.17." 형식
            date_str = str(date_str).replace('.', '-').strip('.')
            return pd.to_datetime(date_str)
    except:
        return None

news_df['parsed_date'] = news_df['date'].apply(parse_date)
news_df['year'] = news_df['parsed_date'].dt.year
news_df['month'] = news_df['parsed_date'].dt.month
news_df['day'] = news_df['parsed_date'].dt.day
news_df['weekday'] = news_df['parsed_date'].dt.weekday  # 0=월요일, 6=일요일
news_df['date_only'] = news_df['parsed_date'].dt.date

print(f"날짜 파싱 성공: {news_df['parsed_date'].notna().sum()}개")
print(f"날짜 범위: {news_df['parsed_date'].min()} ~ {news_df['parsed_date'].max()}")

# ============================================
# 3. 텍스트 피처 추출
# ============================================
print("\n📝 3. 텍스트 피처 추출")
print("-" * 80)

# 제목 길이
news_df['title_length'] = news_df['title'].str.len()

# 요약 길이
news_df['summary_length'] = news_df['summary'].fillna('').str.len()

# 느낌표/물음표 개수 (중요도/의문 지표)
news_df['exclamation_count'] = news_df['title'].str.count('!')
news_df['question_count'] = news_df['title'].str.count('\?')

# 대괄호 개수 (속보, 단독 등)
news_df['bracket_count'] = news_df['title'].str.count('\[|\]')

print(f"제목 평균 길이: {news_df['title_length'].mean():.1f}자")
print(f"요약 평균 길이: {news_df['summary_length'].mean():.1f}자")

# ============================================
# 4. 감성 키워드 분석
# ============================================
print("\n😊 4. 감성 키워드 분석")
print("-" * 80)

# 긍정/부정 키워드 사전
positive_keywords = [
    '성장', '증가', '상승', '호조', '선전', '최고', '기록', '성공', '흑자',
    '급등', '돌파', '확대', '개선', '회복', '강세', '신기록', '최대',
    '선방', '약진', '성과', '효자', '호실적', '대박', '호평', '인기'
]

negative_keywords = [
    '하락', '감소', '부진', '적자', '손실', '악화', '급락', '최저', '부정적',
    '우려', '위기', '하향', '실망', '타격', '논란', '불안', '침체',
    '위축', '부실', '취소', '중단', '지연', '문제', '비판', '악재'
]

neutral_keywords = [
    '출시', '공개', '발표', '계획', '예정', '준비', '개발', '진행', '협력',
    '제휴', '투자', '인수', '합병', '론칭', '업데이트', '서비스'
]

def count_keywords(text, keywords):
    """텍스트에서 키워드 개수 세기"""
    if pd.isna(text):
        return 0
    text = str(text)
    return sum(1 for keyword in keywords if keyword in text)

# 제목과 요약 합친 텍스트
news_df['full_text'] = news_df['title'].fillna('') + ' ' + news_df['summary'].fillna('')

news_df['positive_score'] = news_df['full_text'].apply(
    lambda x: count_keywords(x, positive_keywords)
)
news_df['negative_score'] = news_df['full_text'].apply(
    lambda x: count_keywords(x, negative_keywords)
)
news_df['neutral_score'] = news_df['full_text'].apply(
    lambda x: count_keywords(x, neutral_keywords)
)

# 감성 점수 (-1 ~ 1)
news_df['sentiment_score'] = (
    news_df['positive_score'] - news_df['negative_score']
) / (news_df['positive_score'] + news_df['negative_score'] + 1)

print(f"긍정 키워드 평균: {news_df['positive_score'].mean():.2f}")
print(f"부정 키워드 평균: {news_df['negative_score'].mean():.2f}")
print(f"중립 키워드 평균: {news_df['neutral_score'].mean():.2f}")
print(f"전체 감성 점수 평균: {news_df['sentiment_score'].mean():.3f}")

# ============================================
# 5. 회사별 일일 집계 피처
# ============================================
print("\n🏢 5. 회사별 일일 집계 피처 생성")
print("-" * 80)

# 날짜별, 회사별 집계
daily_features = news_df.groupby(['company', 'date_only']).agg({
    'title': 'count',  # 기사 수
    'positive_score': 'sum',  # 긍정 점수 합
    'negative_score': 'sum',  # 부정 점수 합
    'neutral_score': 'sum',  # 중립 점수 합
    'sentiment_score': 'mean',  # 평균 감성 점수
    'title_length': 'mean',  # 평균 제목 길이
    'summary_length': 'mean',  # 평균 요약 길이
    'exclamation_count': 'sum',  # 느낌표 총 개수
    'question_count': 'sum',  # 물음표 총 개수
    'bracket_count': 'sum',  # 대괄호 총 개수
}).reset_index()

# 컬럼명 변경
daily_features.columns = [
    'company', 'date', 'news_count', 'total_positive', 'total_negative',
    'total_neutral', 'avg_sentiment', 'avg_title_length', 'avg_summary_length',
    'total_exclamation', 'total_question', 'total_bracket'
]

# 추가 피처 생성
daily_features['positive_ratio'] = (
    daily_features['total_positive'] /
    (daily_features['total_positive'] + daily_features['total_negative'] + 1)
)

daily_features['negative_ratio'] = (
    daily_features['total_negative'] /
    (daily_features['total_positive'] + daily_features['total_negative'] + 1)
)

# 뉴스 볼륨 (많은 뉴스 = 중요 이슈)
daily_features['news_volume_score'] = np.log1p(daily_features['news_count'])

print(f"일일 데이터 생성: {len(daily_features)}개 (회사별 날짜별)")
print(f"\n회사별 평균 일일 기사 수:")
print(daily_features.groupby('company')['news_count'].mean().round(1))

# ============================================
# 6. 이동 평균 및 추세 피처
# ============================================
print("\n📈 6. 시계열 피처 생성 (이동평균, 추세)")
print("-" * 80)

def add_time_features(df, company_name, window_sizes=[3, 7, 14]):
    """시계열 피처 추가"""
    company_df = df[df['company'] == company_name].copy()
    company_df = company_df.sort_values('date')

    for window in window_sizes:
        # 뉴스 개수 이동 평균
        company_df[f'news_ma_{window}'] = company_df['news_count'].rolling(
            window=window, min_periods=1
        ).mean()

        # 감성 점수 이동 평균
        company_df[f'sentiment_ma_{window}'] = company_df['avg_sentiment'].rolling(
            window=window, min_periods=1
        ).mean()

        # 긍정/부정 비율 이동 평균
        company_df[f'positive_ratio_ma_{window}'] = company_df['positive_ratio'].rolling(
            window=window, min_periods=1
        ).mean()

    # 전일 대비 변화량
    company_df['news_count_diff'] = company_df['news_count'].diff()
    company_df['sentiment_diff'] = company_df['avg_sentiment'].diff()

    # 주말 여부
    company_df['is_weekend'] = pd.to_datetime(company_df['date']).dt.weekday >= 5

    return company_df

# 각 회사별 시계열 피처 추가
companies = daily_features['company'].unique()
time_series_dfs = []

for company in companies:
    ts_df = add_time_features(daily_features, company)
    time_series_dfs.append(ts_df)
    print(f"  {company}: 시계열 피처 추가 완료")

# 통합
final_features = pd.concat(time_series_dfs, ignore_index=True)

print(f"\n최종 피처 개수: {len(final_features.columns)}개")
print(f"최종 데이터 행 수: {len(final_features)}개")

# ============================================
# 7. 주요 피처 요약
# ============================================
print("\n" + "=" * 80)
print("📊 7. 생성된 주요 피처 목록")
print("=" * 80)

feature_categories = {
    '기본 정보': ['company', 'date'],
    '뉴스 볼륨': ['news_count', 'news_volume_score', 'news_count_diff'],
    '감성 분석': ['avg_sentiment', 'total_positive', 'total_negative',
                 'positive_ratio', 'negative_ratio', 'sentiment_diff'],
    '텍스트 특성': ['avg_title_length', 'avg_summary_length',
                   'total_exclamation', 'total_question', 'total_bracket'],
    '이동평균(3일)': ['news_ma_3', 'sentiment_ma_3', 'positive_ratio_ma_3'],
    '이동평균(7일)': ['news_ma_7', 'sentiment_ma_7', 'positive_ratio_ma_7'],
    '이동평균(14일)': ['news_ma_14', 'sentiment_ma_14', 'positive_ratio_ma_14'],
    '시간 정보': ['is_weekend']
}

for category, features in feature_categories.items():
    print(f"\n[{category}]")
    for feature in features:
        if feature in final_features.columns:
            print(f"  ✓ {feature}")

# ============================================
# 8. 데이터 시각화
# ============================================
print("\n" + "=" * 80)
print("📊 8. 데이터 시각화")
print("=" * 80)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. 회사별 뉴스 개수
company_news = final_features.groupby('company')['news_count'].sum().sort_values(ascending=False)
axes[0, 0].bar(company_news.index, company_news.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0, 0].set_title('회사별 총 뉴스 개수', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('뉴스 개수')
for i, v in enumerate(company_news.values):
    axes[0, 0].text(i, v, f'{int(v)}', ha='center', va='bottom')

# 2. 회사별 평균 감성 점수
company_sentiment = final_features.groupby('company')['avg_sentiment'].mean()
colors = ['green' if x > 0 else 'red' for x in company_sentiment.values]
axes[0, 1].bar(company_sentiment.index, company_sentiment.values, color=colors, alpha=0.7)
axes[0, 1].set_title('회사별 평균 감성 점수', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('감성 점수')
axes[0, 1].axhline(y=0, color='black', linestyle='--', linewidth=0.5)
for i, v in enumerate(company_sentiment.values):
    axes[0, 1].text(i, v, f'{v:.3f}', ha='center', va='bottom' if v > 0 else 'top')

# 3. 시간에 따른 뉴스 개수 추이
for company in companies:
    company_data = final_features[final_features['company'] == company].sort_values('date')
    axes[1, 0].plot(company_data['date'], company_data['news_count'],
                    marker='o', label=company, linewidth=2, markersize=4)
axes[1, 0].set_title('일별 뉴스 개수 추이', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('날짜')
axes[1, 0].set_ylabel('뉴스 개수')
axes[1, 0].legend()
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. 감성 점수 분포
final_features['avg_sentiment'].hist(bins=30, ax=axes[1, 1], color='skyblue', edgecolor='black')
axes[1, 1].set_title('감성 점수 분포', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('감성 점수')
axes[1, 1].set_ylabel('빈도')
axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='중립')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('news_features_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ 시각화 완료! (news_features_analysis.png 저장됨)")

# ============================================
# 9. 상관관계 분석
# ============================================
print("\n" + "=" * 80)
print("🔗 9. 주요 피처 간 상관관계")
print("=" * 80)

# 숫자형 피처만 선택
numeric_features = [
    'news_count', 'avg_sentiment', 'positive_ratio', 'negative_ratio',
    'news_ma_7', 'sentiment_ma_7'
]

corr_matrix = final_features[numeric_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1)
plt.title('피처 간 상관관계 히트맵', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ 상관관계 분석 완료! (correlation_heatmap.png 저장됨)")

# ============================================
# 10. 최종 데이터 저장
# ============================================
print("\n" + "=" * 80)
print("💾 10. 최종 피처 데이터 저장")
print("=" * 80)

# CSV 저장
output_filename = f'news_features_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
final_features.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"✅ 저장 완료: {output_filename}")

# 다운로드
try:
    from google.colab import files
    files.download(output_filename)
    print("✅ 파일 다운로드 시작!")
except:
    print("⚠️ Colab 환경이 아닙니다.")

# ============================================
# 11. 사용 가이드
# ============================================
print("\n" + "=" * 80)
print("📖 11. 생성된 피처 활용 가이드")
print("=" * 80)

print("""
✅ 주가 예측에 유용한 주요 피처:

1. **뉴스 볼륨 지표**
   - news_count: 일일 뉴스 개수 (관심도)
   - news_volume_score: 로그 변환된 뉴스 볼륨
   - news_ma_7: 7일 이동평균 (추세 파악)

2. **감성 분석 지표**
   - avg_sentiment: 평균 감성 점수 (-1 ~ 1)
   - positive_ratio: 긍정 비율
   - sentiment_ma_7: 감성 점수 7일 이동평균

3. **변화량 지표**
   - news_count_diff: 전일 대비 뉴스 개수 변화
   - sentiment_diff: 전일 대비 감성 점수 변화

4. **특수 이벤트 지표**
   - total_exclamation: 느낌표 개수 (속보, 중요 뉴스)
   - total_bracket: 대괄호 개수 (단독, 특종 등)

💡 활용 예시:
   - 머신러닝 모델의 입력 피처로 사용
   - 주가와의 상관관계 분석
   - 시계열 예측 모델의 외생 변수로 활용
   - 감성 점수와 주가 움직임 비교

📊 데이터:
   - 변수명: final_features
   - 형태: {final_features.shape}
   - 기간: {final_features['date'].min()} ~ {final_features['date'].max()}
""")

print("\n" + "=" * 80)
print("✅ 피처 엔지니어링 완료!")
print("=" * 80)
print(f"\n최종 데이터: final_features ({final_features.shape[0]}행 × {final_features.shape[1]}열)")
print("이제 이 데이터를 주가 예측 모델에 활용할 수 있습니다!")

뉴스 데이터 피처 엔지니어링

📊 1. 데이터 확인
--------------------------------------------------------------------------------
전체 뉴스 개수: 0개
컬럼: []

📅 2. 날짜 데이터 전처리
--------------------------------------------------------------------------------


KeyError: 'date'

In [ ]:
# 반복

news_feat_df = aggregate_daily_news_scores(news_df, stock_feature_config)
print("news_feat_df rows:", len(news_feat_df))
news_feat_df.head()

train_with_news_df = build_training_table(price_feat_df, news_feat_df, stock_feature_config)
train_with_news_df[["date", "company", "news_count", "news_score_sum", "future_ret"]].head()


news_feat_df rows: 0


KeyError: 'target'

In [ ]:
print(train_with_news_df["news_count"].value_counts().head())


news_count
0.0    3915
Name: count, dtype: int64


In [ ]:
"""
키워드 기반 뉴스 점수 계산

이미 고쳐놓은 버전의 aggregate_daily_news_scores() 가 있다고 가정하고, 바로 사

"""
news_feat_df = aggregate_daily_news_scores(news_df, stock_feature_config)
news_feat_df.head()


,date,company,news_count,news_score_sum,news_score_avg


In [ ]:
"""
가격 + 뉴스 피처 + 타깃 통합 (새 train_df)

이제 기존 price_feat_df와 방금 만든 news_feat_df를 합쳐서 업그레이드된 학습용 테이블:

"""

train_with_news_df = build_training_table(price_feat_df, news_feat_df, stock_feature_config)

train_with_news_df.head()


,date,open,high,low,close,volume,change,ret_1d,ret_3d,ret_5d,...,ma_5_gap,ma_20_gap,ma_60_gap,company,news_count,news_score_sum,news_score_avg,future_price,future_ret,ret_5_up
0,2021-08-10,448500,480000,400500,454000,5121520,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Krafton,0.0,0.0,0.0,469000.0,0.033040,1
1,2021-08-11,444500,446000,405000,407000,1647759,-0.103524,-0.103524,NaN,NaN,...,NaN,NaN,NaN,Krafton,0.0,0.0,0.0,492500.0,0.210074,1
2,2021-08-12,414000,420500,402000,406000,947958,-0.002457,-0.002457,NaN,NaN,...,NaN,NaN,NaN,Krafton,0.0,0.0,0.0,491500.0,0.210591,1
3,2021-08-13,415000,445500,408500,437000,1669847,0.076355,0.076355,-0.037445,NaN,...,NaN,NaN,NaN,Krafton,0.0,0.0,0.0,484500.0,0.108696,1
4,2021-08-17,433000,460000,423000,451500,1189500,0.033181,0.033181,0.109337,NaN,...,0.047321,NaN,NaN,Krafton,0.0,0.0,0.0,460000.0,0.018826,1


In [ ]:
"""
간단히 “뉴스 피처가 들어갔는지” 확인하는 EDA

예를 들어, 뉴스 많은 날 vs 적은 날의 5일 후 수익률 평균:

"""
import numpy as np

df = train_with_news_df.copy()
df["news_count_bin"] = pd.cut(
    df["news_count"],
    bins=[-0.1, 0, 1, 3, np.inf],
    labels=["0개", "1개", "2~3개", "4개 이상"]
)

group_mean = df.groupby("news_count_bin", observed=False)["future_ret"].mean()
print(group_mean)


news_count_bin
0개       0.001033
1개            NaN
2~3개          NaN
4개 이상         NaN
Name: future_ret, dtype: float64


In [ ]:
group_mean_company = df.groupby(["company", "news_count_bin"], observed=False)["future_ret"].mean()
group_mean_company


company  news_count_bin
Krafton  0개               -0.000447
         1개                     NaN
         2~3개                   NaN
         4개 이상                  NaN
Mgame    0개                0.004584
         1개                     NaN
         2~3개                   NaN
         4개 이상                  NaN
NCSoft   0개               -0.001448
         1개                     NaN
         2~3개                   NaN
         4개 이상                  NaN
Name: future_ret, dtype: float64

In [ ]:
print("news_df rows:", len(news_df))
print("news_feat_df rows:", len(news_feat_df))
print(train_with_news_df["news_count"].value_counts().head())


news_df rows: 0
news_feat_df rows: 0
news_count
0.0    3915
Name: count, dtype: int64
